<a href="https://colab.research.google.com/github/haniyathul/TESIS-HANIY/blob/main/SIMULASI%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## simulasi

# FILE 3 - WEAK SPATIAL DEPENDENCE

## STEP 1 — Grid 10×10

In [17]:
# ============================================================
# FILE 3 - WEAK SPATIAL DEPENDENCE (rho = 0.20)
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Clone repository
# ------------------------------------------------------------

REPO_URL = "https://github.com/haniyathul/TESIS-HANIY.git"
REPO_DIR = Path("/content/TESIS-HANIY")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True
)

# ------------------------------------------------------------
# Folder
# ------------------------------------------------------------

DATA_DIR = REPO_DIR / "data"
ESF_DIR = REPO_DIR / "esf"
WEIGHTS_DIR = REPO_DIR / "weights"

OUTPUT_DIR = Path("/content/FILE3_WEAK")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ SETUP FILE 3 SELESAI")

✅ SETUP FILE 3 SELESAI


## Persiapan data Weak Grid 10–50

In [19]:
# ============================================================
# PERSIAPAN DATA WEAK
# rho = 0.20
# GRID 10 - 50
# ============================================================

GRID_SIZES = [10, 20, 30, 40, 50]

DATA_WEAK = {}


def cari_kolom(df, kandidat):
    """
    Mencari nama kolom tanpa peduli huruf besar/kecil.
    """
    mapping = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for nama in kandidat:
        if nama.lower() in mapping:
            return mapping[nama.lower()]

    raise KeyError(
        f"Kolom tidak ditemukan.\n"
        f"Kandidat: {kandidat}\n"
        f"Kolom tersedia: {df.columns.tolist()}"
    )


for grid in GRID_SIZES:

    # ----------------------------
    # Load data per grid
    # ----------------------------
    file_path = DATA_DIR / f"DATA_GRID{grid}.csv"

    df = pd.read_csv(file_path)

    # ----------------------------
    # Deteksi nama kolom rho
    # ----------------------------
    COL_RHO = cari_kolom(
        df,
        [
            "rho",
            "Rho",
            "RHO",
            "Spatial_Strength_Value",
            "rho_value"
        ]
    )

    COL_SCENARIO = cari_kolom(
        df,
        [
            "Scenario_ID",
            "scenario_id",
            "ScenarioID"
        ]
    )

    # ----------------------------
    # Filter Weak = rho 0.20
    # ----------------------------
    df_weak = df[
        np.isclose(
            pd.to_numeric(
                df[COL_RHO],
                errors="coerce"
            ),
            0.20
        )
    ].copy().reset_index(drop=True)

    DATA_WEAK[grid] = df_weak

    print(
        f"GRID {grid:2d} | "
        f"rho column = {COL_RHO} | "
        f"{len(df_weak):6d} baris | "
        f"{df_weak[COL_SCENARIO].nunique():2d} skenario"
    )


# ============================================================
# NAMA DATA PER GRID
# ============================================================

DATA_WEAK_GRID10 = DATA_WEAK[10]
DATA_WEAK_GRID20 = DATA_WEAK[20]
DATA_WEAK_GRID30 = DATA_WEAK[30]
DATA_WEAK_GRID40 = DATA_WEAK[40]
DATA_WEAK_GRID50 = DATA_WEAK[50]


print("\n✅ SEMUA DATA WEAK BERHASIL DISIAPKAN")

GRID 10 | rho column = Rho |   1800 baris | 18 skenario
GRID 20 | rho column = Rho |   7200 baris | 18 skenario
GRID 30 | rho column = Rho |  16200 baris | 18 skenario
GRID 40 | rho column = Rho |  28800 baris | 18 skenario
GRID 50 | rho column = Rho |  45000 baris | 18 skenario

✅ SEMUA DATA WEAK BERHASIL DISIAPKAN


## HYPERPARAMETER XGBOOST

In [20]:
# ============================================================
# HYPERPARAMETER XGBOOST
# 18 KOMBINASI
# ============================================================

HYPERPARAMETERS = pd.DataFrame([
    ["H01", 200, 0.03, 2],
    ["H02", 200, 0.03, 3],
    ["H03", 200, 0.03, 4],

    ["H04", 200, 0.05, 2],
    ["H05", 200, 0.05, 3],
    ["H06", 200, 0.05, 4],

    ["H07", 200, 0.10, 2],
    ["H08", 200, 0.10, 3],
    ["H09", 200, 0.10, 4],

    ["H10", 300, 0.03, 2],
    ["H11", 300, 0.03, 3],
    ["H12", 300, 0.03, 4],

    ["H13", 300, 0.05, 2],
    ["H14", 300, 0.05, 3],
    ["H15", 300, 0.05, 4],

    ["H16", 300, 0.10, 2],
    ["H17", 300, 0.10, 3],
    ["H18", 300, 0.10, 4],
],
columns=[
    "Hyper_ID",
    "n_estimators",
    "learning_rate",
    "max_depth"
])

display(HYPERPARAMETERS)

,Hyper_ID,n_estimators,learning_rate,max_depth
0,H01,200,0.03,2
1,H02,200,0.03,3
2,H03,200,0.03,4
3,H04,200,0.05,2
4,H05,200,0.05,3
5,H06,200,0.05,4
6,H07,200,0.10,2
7,H08,200,0.10,3
8,H09,200,0.10,4
9,H10,300,0.03,2


In [21]:
# ============================================================
# DAFTAR MODEL
# ============================================================

MODEL_LIST = [
    "XGBoost",
    "ESF-XGBoost Moran",
    "ESF-XGBoost LASSO",
    "ESF-XGBoost Stepwise",
    "ESF-XGBoost SHAP"
]

print("MODEL YANG AKAN DIBANDINGKAN:")
for i, model in enumerate(MODEL_LIST, start=1):
    print(f"{i}. {model}")

MODEL YANG AKAN DIBANDINGKAN:
1. XGBoost
2. ESF-XGBoost Moran
3. ESF-XGBoost LASSO
4. ESF-XGBoost Stepwise
5. ESF-XGBoost SHAP


In [22]:
# ============================================================
# STRUKTUR TABEL EVALUASI / SUMMARY
# ============================================================

SUMMARY_COLUMNS = [

    # Identitas skenario
    "Scenario_ID",
    "Grid_Size",
    "Pattern",
    "W_Method",
    "Error_Structure",
    "Rho",

    # Model
    "Model",
    "ESF_Selection_Method",

    # Hyperparameter terbaik tiap outer fold
    "Best_Hyper_Fold1",
    "Best_Hyper_Fold2",
    "Best_Hyper_Fold3",
    "Best_Hyper_Fold4",
    "Best_Hyper_Fold5",

    # Hyperparameter dominan
    "Dominant_Hyper_ID",
    "Hyper_Frequency",

    # Nilai hyperparameter dominan
    "n_estimators",
    "learning_rate",
    "max_depth",

    # Jumlah EV terpilih
    "N_EV_Fold1",
    "N_EV_Fold2",
    "N_EV_Fold3",
    "N_EV_Fold4",
    "N_EV_Fold5",
    "Mean_N_EV_Selected",

    # Evaluasi prediksi
    "R2_OOF",
    "RMSE_OOF",
    "MAE_OOF",

    # Evaluasi spasial residual
    "Moran_I_Residual_OOF",
    "Moran_pvalue_Residual_OOF",

    # Status
    "Status"
]

SUMMARY_TEMPLATE = pd.DataFrame(columns=SUMMARY_COLUMNS)

print("✅ Struktur tabel evaluasi siap")
display(SUMMARY_TEMPLATE)

✅ Struktur tabel evaluasi siap


,Scenario_ID,Grid_Size,Pattern,W_Method,Error_Structure,Rho,Model,ESF_Selection_Method,Best_Hyper_Fold1,Best_Hyper_Fold2,Best_Hyper_Fold3,Best_Hyper_Fold4,Best_Hyper_Fold5,Dominant_Hyper_ID,Hyper_Frequency,n_estimators,learning_rate,max_depth,N_EV_Fold1,N_EV_Fold2,N_EV_Fold3,N_EV_Fold4,N_EV_Fold5,Mean_N_EV_Selected,R2_OOF,RMSE_OOF,MAE_OOF,Moran_I_Residual_OOF,Moran_pvalue_Residual_OOF,Status


##

In [23]:
# ============================================================
# STRUKTUR DETAIL PREDIKSI
# ============================================================

DETAIL_COLUMNS = [
    "Scenario_ID",
    "Location_ID",
    "x",
    "y",
    "Y_Actual",
    "Outer_Fold",

    "Pred_XGBoost",
    "Residual_XGBoost",

    "Pred_ESF_Moran",
    "Residual_ESF_Moran",

    "Pred_ESF_LASSO",
    "Residual_ESF_LASSO",

    "Pred_ESF_Stepwise",
    "Residual_ESF_Stepwise",

    "Pred_ESF_SHAP",
    "Residual_ESF_SHAP"
]

DETAIL_TEMPLATE = pd.DataFrame(columns=DETAIL_COLUMNS)

print("✅ Struktur detail prediksi siap")
display(DETAIL_TEMPLATE)

✅ Struktur detail prediksi siap


,Scenario_ID,Location_ID,x,y,Y_Actual,Outer_Fold,Pred_XGBoost,Residual_XGBoost,Pred_ESF_Moran,Residual_ESF_Moran,Pred_ESF_LASSO,Residual_ESF_LASSO,Pred_ESF_Stepwise,Residual_ESF_Stepwise,Pred_ESF_SHAP,Residual_ESF_SHAP


# MODELING GRID 10 X 10 RHO WEAK

In [25]:
# ============================================================
# PARAMETER XGBOOST YANG TETAP
# ============================================================

XGB_FIXED_PARAMS = {
    "objective": "reg:squarederror",
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "random_state": 2026,
    "n_jobs": -1
}

print("✅ XGB_FIXED_PARAMS siap")

✅ XGB_FIXED_PARAMS siap


In [26]:
# ============================================================
# FILE 3 - WEAK
# MODELING GRID 10
# PERSIAPAN MODEL
# ============================================================

import sys
import subprocess
import warnings
import re
import os
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.cluster import KMeans
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LassoCV,
    LinearRegression
)

# ------------------------------------------------------------
# Pastikan XGBoost tersedia
# ------------------------------------------------------------

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "xgboost", "openpyxl"
    ])
    import xgboost as xgb
    from xgboost import XGBRegressor


# ============================================================
# KONFIGURASI
# ============================================================

GRID = 10
RHO_WEAK = 0.20

OUTER_FOLDS = 5
INNER_FOLDS = 3

SEED_MODEL = 2026

# Threshold kandidat Moran eigenvector
MORAN_EV_THRESHOLD = 0.25

# SHAP: ambil EV sampai cumulative importance 90%
SHAP_CUMULATIVE = 0.90

# Stepwise BIC
BIC_MIN_IMPROVEMENT = 2.0

# Moran permutation
MORAN_PERMUTATIONS = 999


# ============================================================
# CEK OBJEK DARI CELL SEBELUMNYA
# ============================================================

assert "DATA_WEAK_GRID10" in globals(), \
    "DATA_WEAK_GRID10 belum tersedia"

assert "HYPERPARAMETERS" in globals(), \
    "HYPERPARAMETERS belum tersedia"

assert "XGB_FIXED_PARAMS" in globals(), \
    "XGB_FIXED_PARAMS belum tersedia"

assert len(HYPERPARAMETERS) == 18, \
    "Hyperparameter harus H01-H18"


DATA_MODEL = DATA_WEAK_GRID10.copy()

print("✅ DATA GRID10 :", DATA_MODEL.shape)
print("✅ Hyperparameter :", len(HYPERPARAMETERS))
print("✅ Modeling setup siap")

✅ DATA GRID10 : (1800, 37)
✅ Hyperparameter : 18
✅ Modeling setup siap


In [27]:
# ============================================================
# LOAD W + ESF GRID 10
# ============================================================

def canonical_w(value):

    s = str(value).upper()

    if "KNN" in s:
        return "KNN5"

    if "IDW" in s:
        return "IDW"

    if "DIST" in s:
        return "DISTANCE"

    raise ValueError(
        f"W method tidak dikenali: {value}"
    )


def find_npz_resource(folder, grid, method):

    folder = Path(folder)

    files = list(folder.glob("*.npz"))

    grid_token = f"GRID{grid}"

    hasil = []

    for f in files:

        name = f.name.upper()

        if grid_token not in name:
            continue

        if method == "KNN5":
            cocok = "KNN" in name

        elif method == "IDW":
            cocok = "IDW" in name

        elif method == "DISTANCE":
            cocok = (
                "DIST" in name
                and "IDW" not in name
            )

        else:
            cocok = False

        if cocok:
            hasil.append(f)

    if len(hasil) == 0:
        raise FileNotFoundError(
            f"Tidak menemukan {method} Grid{grid} "
            f"di {folder}"
        )

    # Pilih nama paling pendek jika ada lebih dari 1
    hasil = sorted(
        hasil,
        key=lambda x: len(x.name)
    )

    return hasil[0]


def load_esf_matrix(path, n):

    with np.load(path, allow_pickle=False) as z:

        candidates = []

        for key in z.files:

            arr = np.asarray(z[key])

            if not np.issubdtype(
                arr.dtype,
                np.number
            ):
                continue

            if arr.ndim != 2:
                continue

            # n x k
            if arr.shape[0] == n:
                E = arr.astype(float)

            # k x n
            elif arr.shape[1] == n:
                E = arr.T.astype(float)

            else:
                continue

            score = 0

            key_low = key.lower()

            if "eigenvector" in key_low:
                score += 100

            if "eigvec" in key_low:
                score += 80

            if key_low in ["ev", "e", "vectors"]:
                score += 60

            if E.shape[1] == n - 1:
                score += 50

            candidates.append(
                (score, key, E)
            )

    if len(candidates) == 0:

        raise ValueError(
            f"Tidak menemukan matriks eigenvector "
            f"dalam {path.name}"
        )

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    score, key, E = candidates[0]

    return E, key


# ============================================================
# LOAD 3 W DAN 3 ESF
# ============================================================

W_GRID10 = {}
ESF_GRID10 = {}
EIGVAL_GRID10 = {}

for method in [
    "KNN5",
    "DISTANCE",
    "IDW"
]:

    # ----------------------------
    # W
    # ----------------------------

    w_file = find_npz_resource(
        WEIGHTS_DIR,
        GRID,
        method
    )

    W = sparse.load_npz(
        w_file
    ).tocsr()

    # ----------------------------
    # ESF
    # ----------------------------

    esf_file = find_npz_resource(
        ESF_DIR,
        GRID,
        method
    )

    E, esf_key = load_esf_matrix(
        esf_file,
        n=GRID * GRID
    )

    # ----------------------------
    # Hitung eigenvalue langsung
    # dari W symmetric
    # ----------------------------

    W_sym = (
        W + W.T
    ) * 0.5

    WE = W_sym @ E

    numerator = np.sum(
        E * WE,
        axis=0
    )

    denominator = np.sum(
        E * E,
        axis=0
    )

    eigenvalues = (
        numerator /
        np.maximum(
            denominator,
            1e-15
        )
    )

    W_GRID10[method] = W
    ESF_GRID10[method] = E
    EIGVAL_GRID10[method] = eigenvalues

    print(
        f"{method:10s} | "
        f"W={W.shape} | "
        f"ESF={E.shape} | "
        f"key={esf_key}"
    )


print("\n✅ Semua W + ESF Grid10 berhasil dimuat")

KNN5       | W=(100, 100) | ESF=(100, 99) | key=Eigenvectors
DISTANCE   | W=(100, 100) | ESF=(100, 99) | key=Eigenvectors
IDW        | W=(100, 100) | ESF=(100, 99) | key=Eigenvectors

✅ Semua W + ESF Grid10 berhasil dimuat


In [30]:
# ============================================================
# INIT MODELING GRID 10
# Deteksi semua kolom yang dibutuhkan
# ============================================================

DATA_MODEL = DATA_WEAK_GRID10.copy()


def find_col(df, candidates, required=True, exclude=None):
    if exclude is None:
        exclude = []

    mapping = {
        str(c).strip().lower(): c
        for c in df.columns
        if c not in exclude
    }

    for candidate in candidates:
        key = str(candidate).strip().lower()

        if key in mapping:
            return mapping[key]

    if required:
        raise KeyError(
            f"\nKolom tidak ditemukan.\n"
            f"Kandidat: {candidates}\n\n"
            f"Kolom tersedia:\n{df.columns.tolist()}"
        )

    return None


# ============================================================
# IDENTITAS SCENARIO
# ============================================================

COL_SCENARIO = find_col(
    DATA_MODEL,
    [
        "Scenario_ID",
        "ScenarioID",
        "scenario_id"
    ]
)

COL_PATTERN = find_col(
    DATA_MODEL,
    [
        "Pattern",
        "Spatial_Pattern",
        "Pattern_Type"
    ]
)

COL_W = find_col(
    DATA_MODEL,
    [
        "W_Method",
        "Weight_Method",
        "W_Type",
        "W"
    ]
)

COL_ERROR = find_col(
    DATA_MODEL,
    [
        "Error_Structure",
        "Error_Type",
        "Error"
    ]
)

COL_RHO = find_col(
    DATA_MODEL,
    [
        "rho",
        "Rho",
        "RHO",
        "Spatial_Strength_Value",
        "rho_value"
    ],
    required=False
)


# ============================================================
# TARGET Y
# ============================================================

COL_TARGET = find_col(
    DATA_MODEL,
    [
        "Y",
        "Y_Response",
        "Response",
        "Target"
    ]
)


# ============================================================
# KOORDINAT
# ============================================================

COL_XCOORD = find_col(
    DATA_MODEL,
    [
        "x",
        "x_norm",
        "X_norm",
        "x_coord",
        "X_coord",
        "coord_x",
        "grid_x",
        "Grid_X"
    ],
    exclude=[COL_TARGET]
)

COL_YCOORD = find_col(
    DATA_MODEL,
    [
        "y",
        "y_norm",
        "Y_norm",
        "y_coord",
        "Y_coord",
        "coord_y",
        "grid_y",
        "Grid_Y"
    ],
    exclude=[COL_TARGET]
)


# ============================================================
# LOCATION ID
# ============================================================

COL_LOCATION = find_col(
    DATA_MODEL,
    [
        "Location_ID",
        "Location",
        "Loc_ID",
        "ID_Location"
    ],
    required=False
)


# ============================================================
# 20 PREDICTOR
# X_spatial + X1 ... X19
# ============================================================

PREDICTOR_COLS = []

for nama in ["X_spatial"] + [f"X{i}" for i in range(1, 20)]:

    col = find_col(
        DATA_MODEL,
        [nama]
    )

    PREDICTOR_COLS.append(col)


# ============================================================
# VALIDASI
# ============================================================

assert len(PREDICTOR_COLS) == 20
assert DATA_MODEL[COL_SCENARIO].nunique() == 18
assert len(DATA_MODEL) == 1800


print("=" * 65)
print("✅ INIT MODELING GRID10 BERHASIL")
print("=" * 65)

print("Scenario :", COL_SCENARIO)
print("Pattern  :", COL_PATTERN)
print("W Method :", COL_W)
print("Error    :", COL_ERROR)
print("Rho      :", COL_RHO)
print("Target Y :", COL_TARGET)
print("X Coord  :", COL_XCOORD)
print("Y Coord  :", COL_YCOORD)
print("Location :", COL_LOCATION)

print("\nJumlah predictor :", len(PREDICTOR_COLS))
print("Predictor:")
print(PREDICTOR_COLS)

print("\nJumlah baris     :", len(DATA_MODEL))
print("Jumlah skenario  :", DATA_MODEL[COL_SCENARIO].nunique())

✅ INIT MODELING GRID10 BERHASIL
Scenario : Scenario_ID
Pattern  : Pattern
W Method : W_Method
Error    : Error_Structure
Rho      : Rho
Target Y : Y
X Coord  : X_norm
Y Coord  : Y_norm
Location : Location_ID

Jumlah predictor : 20
Predictor:
['X_spatial', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19']

Jumlah baris     : 1800
Jumlah skenario  : 18


In [32]:
# ============================================================
# 5-FOLD SPATIAL CROSS VALIDATION
# GRID 10
# ============================================================

from sklearn.cluster import KMeans
import numpy as np

OUTER_FOLDS = 5
INNER_FOLDS = 3
SEED_MODEL = 2026


# ============================================================
# Ambil 1 skenario sebagai referensi koordinat
# ============================================================

FIRST_SCENARIO = (
    DATA_MODEL[COL_SCENARIO]
    .drop_duplicates()
    .iloc[0]
)

REF_GRID10 = (
    DATA_MODEL[
        DATA_MODEL[COL_SCENARIO] == FIRST_SCENARIO
    ]
    .copy()
    .sort_values(
        [COL_YCOORD, COL_XCOORD]
    )
    .reset_index(drop=True)
)

COORDS_GRID10 = (
    REF_GRID10[
        [COL_XCOORD, COL_YCOORD]
    ]
    .to_numpy(dtype=float)
)

assert COORDS_GRID10.shape == (100, 2)


# ============================================================
# KMEANS UNTUK MEMBENTUK 5 BLOK SPASIAL
# ============================================================

kmeans_outer = KMeans(
    n_clusters=OUTER_FOLDS,
    random_state=SEED_MODEL,
    n_init=20
)

OUTER_LABELS_GRID10 = (
    kmeans_outer.fit_predict(
        COORDS_GRID10
    )
)


# ============================================================
# CEK JUMLAH LOKASI PER FOLD
# ============================================================

print("=" * 55)
print("5 OUTER SPATIAL FOLDS - GRID10")
print("=" * 55)

for fold in range(OUTER_FOLDS):

    n_test = np.sum(
        OUTER_LABELS_GRID10 == fold
    )

    n_train = 100 - n_test

    print(
        f"Fold {fold + 1} | "
        f"Train = {n_train:2d} | "
        f"Test = {n_test:2d}"
    )


assert len(np.unique(OUTER_LABELS_GRID10)) == 5
assert len(OUTER_LABELS_GRID10) == 100

print("\n✅ 5 SPATIAL FOLD GRID10 SIAP")

5 OUTER SPATIAL FOLDS - GRID10
Fold 1 | Train = 84 | Test = 16
Fold 2 | Train = 79 | Test = 21
Fold 3 | Train = 81 | Test = 19
Fold 4 | Train = 78 | Test = 22
Fold 5 | Train = 78 | Test = 22

✅ 5 SPATIAL FOLD GRID10 SIAP


## Baseline XGBoost + 18 hyperparameter

In [33]:
# ============================================================
# BASELINE XGBOOST
# WEAK - GRID 10
# 18 SCENARIO
# 5 OUTER SPATIAL FOLD
# 3 INNER SPATIAL FOLD
# H01 - H18
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.cluster import KMeans
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from xgboost import XGBRegressor


# ============================================================
# HELPER
# ============================================================

def hitung_rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def buat_model_xgb(hp):

    params = XGB_FIXED_PARAMS.copy()

    params.update({
        "n_estimators":
            int(hp["n_estimators"]),

        "learning_rate":
            float(hp["learning_rate"]),

        "max_depth":
            int(hp["max_depth"])
    })

    return XGBRegressor(**params)


# ============================================================
# INNER SPATIAL SPLIT
# ============================================================

def buat_inner_split(
    coords_train,
    n_splits=3,
    seed=2026
):

    kmeans = KMeans(
        n_clusters=n_splits,
        random_state=seed,
        n_init=20
    )

    labels = kmeans.fit_predict(
        coords_train
    )

    splits = []

    for fold in range(n_splits):

        val_idx = np.where(
            labels == fold
        )[0]

        train_idx = np.where(
            labels != fold
        )[0]

        splits.append(
            (train_idx, val_idx)
        )

    return splits


# ============================================================
# CARI HYPERPARAMETER TERBAIK
# ============================================================

def cari_hyperparameter_terbaik(
    X_train,
    y_train,
    inner_splits
):

    hasil_tuning = []

    for _, hp in HYPERPARAMETERS.iterrows():

        fold_rmse = []

        for inner_train, inner_val in inner_splits:

            model = buat_model_xgb(hp)

            model.fit(
                X_train[inner_train],
                y_train[inner_train]
            )

            pred = model.predict(
                X_train[inner_val]
            )

            fold_rmse.append(
                hitung_rmse(
                    y_train[inner_val],
                    pred
                )
            )

        hasil_tuning.append({

            "Hyper_ID":
                hp["Hyper_ID"],

            "n_estimators":
                int(hp["n_estimators"]),

            "learning_rate":
                float(hp["learning_rate"]),

            "max_depth":
                int(hp["max_depth"]),

            "Inner_CV_RMSE":
                float(
                    np.mean(fold_rmse)
                )
        })


    hasil_tuning = pd.DataFrame(
        hasil_tuning
    )

    hasil_tuning = (
        hasil_tuning
        .sort_values([
            "Inner_CV_RMSE",
            "Hyper_ID"
        ])
        .reset_index(drop=True)
    )

    best = (
        hasil_tuning
        .iloc[0]
        .to_dict()
    )

    return best, hasil_tuning


# ============================================================
# STORAGE
# ============================================================

BASELINE_SUMMARY = []
BASELINE_FOLD_RESULTS = []
BASELINE_PREDICTIONS = {}

SCENARIO_IDS = (
    DATA_MODEL[COL_SCENARIO]
    .drop_duplicates()
    .tolist()
)


print("=" * 75)
print("BASELINE XGBOOST - WEAK GRID10")
print("=" * 75)

print("Scenario :", len(SCENARIO_IDS))
print("Outer CV :", OUTER_FOLDS)
print("Inner CV :", INNER_FOLDS)
print("Hyper    :", len(HYPERPARAMETERS))

print("=" * 75)


# ============================================================
# LOOP 18 SCENARIO
# ============================================================

for nomor, scenario_id in enumerate(
    SCENARIO_IDS,
    start=1
):

    # --------------------------------------------------------
    # Ambil 1 scenario
    # --------------------------------------------------------

    df_s = (
        DATA_MODEL[
            DATA_MODEL[COL_SCENARIO]
            == scenario_id
        ]
        .copy()
        .sort_values([
            COL_YCOORD,
            COL_XCOORD
        ])
        .reset_index(drop=True)
    )

    assert len(df_s) == 100


    # --------------------------------------------------------
    # X, Y, koordinat
    # --------------------------------------------------------

    X = (
        df_s[PREDICTOR_COLS]
        .to_numpy(dtype=float)
    )

    y = (
        df_s[COL_TARGET]
        .to_numpy(dtype=float)
    )

    coords = (
        df_s[
            [COL_XCOORD, COL_YCOORD]
        ]
        .to_numpy(dtype=float)
    )


    # Pastikan posisi sama dengan spatial fold reference
    assert np.allclose(
        coords,
        COORDS_GRID10
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    pattern = (
        df_s[COL_PATTERN]
        .iloc[0]
    )

    w_method = (
        df_s[COL_W]
        .iloc[0]
    )

    error_type = (
        df_s[COL_ERROR]
        .iloc[0]
    )

    rho_value = (
        float(
            df_s[COL_RHO]
            .iloc[0]
        )
    )


    # --------------------------------------------------------
    # OOF prediction
    # --------------------------------------------------------

    oof_pred = np.full(
        100,
        np.nan
    )

    best_hyper_fold = []
    inner_rmse_fold = []


    print(
        f"\n[{nomor:02d}/18] "
        f"{scenario_id}"
    )


    # ========================================================
    # 5 OUTER FOLD
    # ========================================================

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_idx = np.where(
            OUTER_LABELS_GRID10
            == outer_fold
        )[0]

        train_idx = np.where(
            OUTER_LABELS_GRID10
            != outer_fold
        )[0]


        X_train = X[train_idx]
        y_train = y[train_idx]

        X_test = X[test_idx]
        y_test = y[test_idx]

        coords_train = coords[
            train_idx
        ]


        # ====================================================
        # INNER 3-FOLD SPATIAL CV
        # ====================================================

        inner_splits = (
            buat_inner_split(
                coords_train,
                n_splits=INNER_FOLDS,
                seed=
                    SEED_MODEL
                    + outer_fold
            )
        )


        # ====================================================
        # COBA H01-H18
        # ====================================================

        best_hp, tuning_table = (
            cari_hyperparameter_terbaik(
                X_train,
                y_train,
                inner_splits
            )
        )


        # ====================================================
        # FIT BEST MODEL
        # seluruh outer training
        # ====================================================

        best_model = (
            buat_model_xgb(
                best_hp
            )
        )

        best_model.fit(
            X_train,
            y_train
        )


        pred_test = (
            best_model.predict(
                X_test
            )
        )


        # Simpan OOF prediction
        oof_pred[
            test_idx
        ] = pred_test


        best_hyper_fold.append(
            best_hp["Hyper_ID"]
        )

        inner_rmse_fold.append(
            best_hp[
                "Inner_CV_RMSE"
            ]
        )


        # ====================================================
        # EVALUASI FOLD
        # ====================================================

        fold_r2 = r2_score(
            y_test,
            pred_test
        )

        fold_rmse = hitung_rmse(
            y_test,
            pred_test
        )

        fold_mae = mean_absolute_error(
            y_test,
            pred_test
        )


        BASELINE_FOLD_RESULTS.append({

            "Scenario_ID":
                scenario_id,

            "Grid_Size":
                10,

            "Pattern":
                pattern,

            "W_Method":
                w_method,

            "Error_Structure":
                error_type,

            "Rho":
                rho_value,

            "Model":
                "XGBoost",

            "Outer_Fold":
                outer_fold + 1,

            "Best_Hyper_ID":
                best_hp[
                    "Hyper_ID"
                ],

            "n_estimators":
                int(
                    best_hp[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    best_hp[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    best_hp[
                        "max_depth"
                    ]
                ),

            "Inner_CV_RMSE":
                float(
                    best_hp[
                        "Inner_CV_RMSE"
                    ]
                ),

            "R2_Test":
                float(fold_r2),

            "RMSE_Test":
                float(fold_rmse),

            "MAE_Test":
                float(fold_mae)
        })


        print(
            f"   Fold {outer_fold + 1} "
            f"→ Best = "
            f"{best_hp['Hyper_ID']} | "
            f"Inner RMSE = "
            f"{best_hp['Inner_CV_RMSE']:.4f}"
        )


    # ========================================================
    # VALIDASI OOF
    # ========================================================

    assert np.isfinite(
        oof_pred
    ).all()


    # ========================================================
    # EVALUASI OOF 100 LOKASI
    # ========================================================

    R2_OOF = r2_score(
        y,
        oof_pred
    )

    RMSE_OOF = hitung_rmse(
        y,
        oof_pred
    )

    MAE_OOF = mean_absolute_error(
        y,
        oof_pred
    )


    # ========================================================
    # HYPERPARAMETER DOMINAN
    # ========================================================

    counter = Counter(
        best_hyper_fold
    )

    max_freq = max(
        counter.values()
    )

    kandidat_dominan = [
        h
        for h, f
        in counter.items()
        if f == max_freq
    ]


    # jika seri → gunakan yang mean inner RMSE lebih kecil
    if len(
        kandidat_dominan
    ) == 1:

        dominant_hyper = (
            kandidat_dominan[0]
        )

    else:

        mean_rmse_by_hyper = {}

        for h in kandidat_dominan:

            nilai = [
                BASELINE_FOLD_RESULTS[-5 + i][
                    "Inner_CV_RMSE"
                ]
                for i in range(5)
                if best_hyper_fold[i] == h
            ]

            mean_rmse_by_hyper[h] = (
                np.mean(nilai)
            )

        dominant_hyper = min(
            kandidat_dominan,
            key=lambda h:
                mean_rmse_by_hyper[h]
        )


    dominant_row = (
        HYPERPARAMETERS[
            HYPERPARAMETERS[
                "Hyper_ID"
            ] == dominant_hyper
        ]
        .iloc[0]
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    BASELINE_SUMMARY.append({

        "Scenario_ID":
            scenario_id,

        "Grid_Size":
            10,

        "Pattern":
            pattern,

        "W_Method":
            w_method,

        "Error_Structure":
            error_type,

        "Rho":
            rho_value,

        "Model":
            "XGBoost",

        "ESF_Selection_Method":
            "None",

        "Best_Hyper_Fold1":
            best_hyper_fold[0],

        "Best_Hyper_Fold2":
            best_hyper_fold[1],

        "Best_Hyper_Fold3":
            best_hyper_fold[2],

        "Best_Hyper_Fold4":
            best_hyper_fold[3],

        "Best_Hyper_Fold5":
            best_hyper_fold[4],

        "Dominant_Hyper_ID":
            dominant_hyper,

        "Hyper_Frequency":
            f"{max_freq}/5",

        "n_estimators":
            int(
                dominant_row[
                    "n_estimators"
                ]
            ),

        "learning_rate":
            float(
                dominant_row[
                    "learning_rate"
                ]
            ),

        "max_depth":
            int(
                dominant_row[
                    "max_depth"
                ]
            ),

        "N_EV_Fold1": 0,
        "N_EV_Fold2": 0,
        "N_EV_Fold3": 0,
        "N_EV_Fold4": 0,
        "N_EV_Fold5": 0,

        "Mean_N_EV_Selected":
            0.0,

        "Mean_Inner_CV_RMSE":
            float(
                np.mean(
                    inner_rmse_fold
                )
            ),

        "R2_OOF":
            float(R2_OOF),

        "RMSE_OOF":
            float(RMSE_OOF),

        "MAE_OOF":
            float(MAE_OOF),

        "Status":
            "OK"
    })


    # ========================================================
    # SIMPAN PREDIKSI
    # ========================================================

    BASELINE_PREDICTIONS[
        scenario_id
    ] = pd.DataFrame({

        "Scenario_ID":
            scenario_id,

        "Location_ID":
            (
                df_s[COL_LOCATION]
                .to_numpy()
                if COL_LOCATION is not None
                else
                np.arange(1, 101)
            ),

        "X_norm":
            coords[:, 0],

        "Y_norm":
            coords[:, 1],

        "Y_Actual":
            y,

        "Outer_Fold":
            OUTER_LABELS_GRID10 + 1,

        "Pred_XGBoost":
            oof_pred,

        "Residual_XGBoost":
            y - oof_pred
    })


    print(
        f"   ✅ OOF: "
        f"R2={R2_OOF:.4f} | "
        f"RMSE={RMSE_OOF:.4f} | "
        f"MAE={MAE_OOF:.4f} | "
        f"Dominan={dominant_hyper}"
    )


# ============================================================
# DATAFRAME FINAL BASELINE
# ============================================================

BASELINE_SUMMARY_GRID10 = (
    pd.DataFrame(
        BASELINE_SUMMARY
    )
)

BASELINE_FOLD_GRID10 = (
    pd.DataFrame(
        BASELINE_FOLD_RESULTS
    )
)


print("\n" + "=" * 75)
print("🎉 BASELINE XGBOOST GRID10 SELESAI")
print("=" * 75)

print(
    "Summary:",
    BASELINE_SUMMARY_GRID10.shape
)

print(
    "Fold Evaluation:",
    BASELINE_FOLD_GRID10.shape
)

print(
    "Prediction scenario:",
    len(BASELINE_PREDICTIONS)
)

BASELINE XGBOOST - WEAK GRID10
Scenario : 18
Outer CV : 5
Inner CV : 3
Hyper    : 18

[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS
   Fold 1 → Best = H10 | Inner RMSE = 2.0805
   Fold 2 → Best = H02 | Inner RMSE = 2.3482
   Fold 3 → Best = H13 | Inner RMSE = 2.2037
   Fold 4 → Best = H04 | Inner RMSE = 2.1394
   Fold 5 → Best = H04 | Inner RMSE = 2.0939
   ✅ OOF: R2=0.3338 | RMSE=2.6180 | MAE=1.9453 | Dominan=H04

[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS
   Fold 1 → Best = H10 | Inner RMSE = 2.0454
   Fold 2 → Best = H09 | Inner RMSE = 2.3561
   Fold 3 → Best = H14 | Inner RMSE = 2.1841
   Fold 4 → Best = H13 | Inner RMSE = 2.0874
   Fold 5 → Best = H04 | Inner RMSE = 2.1414
   ✅ OOF: R2=0.2993 | RMSE=2.6537 | MAE=1.9919 | Dominan=H10

[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS
   Fold 1 → Best = H10 | Inner RMSE = 2.0999
   Fold 2 → Best = H12 | Inner RMSE = 2.3545
   Fold 3 → Best = H10 | Inner RMSE = 2.2016
   Fold 4 → Best = H04 | Inner RMSE = 2.1463
   Fold 5 → Best = H10 

In [34]:
# ============================================================
# VALIDASI BASELINE GRID10
# ============================================================

assert len(
    BASELINE_SUMMARY_GRID10
) == 18

assert len(
    BASELINE_FOLD_GRID10
) == 90

assert len(
    BASELINE_PREDICTIONS
) == 18

assert (
    BASELINE_SUMMARY_GRID10[
        "R2_OOF"
    ]
    .notna()
    .all()
)

assert (
    BASELINE_SUMMARY_GRID10[
        "RMSE_OOF"
    ]
    .notna()
    .all()
)

assert (
    BASELINE_SUMMARY_GRID10[
        "MAE_OOF"
    ]
    .notna()
    .all()
)


print("=" * 65)
print("✅ BASELINE XGBOOST GRID10: PASS")
print("=" * 65)

display(
    BASELINE_SUMMARY_GRID10[
        [
            "Scenario_ID",
            "Pattern",
            "W_Method",
            "Error_Structure",

            "Best_Hyper_Fold1",
            "Best_Hyper_Fold2",
            "Best_Hyper_Fold3",
            "Best_Hyper_Fold4",
            "Best_Hyper_Fold5",

            "Dominant_Hyper_ID",
            "Hyper_Frequency",

            "R2_OOF",
            "RMSE_OOF",
            "MAE_OOF"
        ]
    ]
)

✅ BASELINE XGBOOST GRID10: PASS


,Scenario_ID,Pattern,W_Method,Error_Structure,Best_Hyper_Fold1,Best_Hyper_Fold2,Best_Hyper_Fold3,Best_Hyper_Fold4,Best_Hyper_Fold5,Dominant_Hyper_ID,Hyper_Frequency,R2_OOF,RMSE_OOF,MAE_OOF
0,G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS,Hotspot,KNN-5,Homogeneous,H10,H02,H13,H04,H04,H04,2/5,0.333753,2.617963,1.945317
1,G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS,Hotspot,KNN-5,Heterogeneous,H10,H09,H14,H13,H04,H10,1/5,0.299337,2.653730,1.991911
2,G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS,Hotspot,Distance Band,Homogeneous,H10,H12,H10,H04,H10,H10,3/5,0.321421,2.638625,1.966141
3,G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS,Hotspot,Distance Band,Heterogeneous,H10,H06,H17,H13,H10,H10,2/5,0.293824,2.660471,2.003745
4,G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS,Hotspot,IDW,Homogeneous,H10,H01,H10,H10,H07,H10,3/5,0.331859,2.559854,1.883915
5,G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS,Hotspot,IDW,Heterogeneous,H10,H01,H13,H04,H04,H04,2/5,0.318976,2.553142,1.922072
6,G10_KORIDOR_KNN5_WEAK_HOMOGENEOUS,Koridor,KNN-5,Homogeneous,H12,H11,H04,H10,H09,H10,1/5,0.513020,2.096552,1.647572
7,G10_KORIDOR_KNN5_WEAK_HETEROGENEOUS,Koridor,KNN-5,Heterogeneous,H12,H05,H04,H10,H08,H10,1/5,0.491849,2.121398,1.730957
8,G10_KORIDOR_DISTANCE_BAND_WEAK_HOMOGENEOUS,Koridor,Distance Band,Homogeneous,H15,H11,H04,H10,H08,H10,1/5,0.527318,2.066652,1.641310
9,G10_KORIDOR_DISTANCE_BAND_WEAK_HETEROGENEOUS,Koridor,Distance Band,Heterogeneous,H12,H05,H13,H10,H18,H10,1/5,0.507843,2.088662,1.721827


## ESF-XGBoost Moran

In [35]:
# ============================================================
# ESF-XGBOOST MORAN
# WEAK - GRID10
# ============================================================

from collections import Counter

# Pastikan hasil File 2 sudah tersedia
assert "ESF_GRID10" in globals(), "ESF_GRID10 belum tersedia"
assert "EIGVAL_GRID10" in globals(), "EIGVAL_GRID10 belum tersedia"

MORAN_THRESHOLD = 0.25


# ============================================================
# SAMAKAN NAMA W
# ============================================================

def canonical_w(value):

    s = str(value).upper()

    if "KNN" in s:
        return "KNN5"

    if "IDW" in s:
        return "IDW"

    if "DIST" in s:
        return "DISTANCE"

    raise ValueError(f"W tidak dikenali: {value}")


# ============================================================
# PILIH EV MORAN
# positive eigenvalue >= 25% eigenvalue positif maksimum
# ============================================================

def pilih_ev_moran(eigenvalues, threshold=0.25):

    eigenvalues = np.asarray(
        eigenvalues,
        dtype=float
    )

    positif = np.where(
        eigenvalues > 0
    )[0]

    if len(positif) == 0:
        return np.array([], dtype=int)

    batas = (
        threshold
        * eigenvalues[positif].max()
    )

    selected = positif[
        eigenvalues[positif] >= batas
    ]

    # urut terbesar
    selected = selected[
        np.argsort(
            eigenvalues[selected]
        )[::-1]
    ]

    return selected


# ============================================================
# STORAGE
# ============================================================

MORAN_SUMMARY = []
MORAN_FOLD_RESULTS = []
MORAN_PREDICTIONS = {}


# ============================================================
# LOOP 18 SCENARIO
# ============================================================

for nomor, scenario_id in enumerate(
    SCENARIO_IDS,
    start=1
):

    df_s = (
        DATA_MODEL[
            DATA_MODEL[COL_SCENARIO] == scenario_id
        ]
        .copy()
        .sort_values(
            [COL_YCOORD, COL_XCOORD]
        )
        .reset_index(drop=True)
    )

    X_base = (
        df_s[PREDICTOR_COLS]
        .to_numpy(dtype=float)
    )

    y = (
        df_s[COL_TARGET]
        .to_numpy(dtype=float)
    )

    coords = (
        df_s[
            [COL_XCOORD, COL_YCOORD]
        ]
        .to_numpy(dtype=float)
    )

    pattern = df_s[COL_PATTERN].iloc[0]
    w_raw = df_s[COL_W].iloc[0]
    error_type = df_s[COL_ERROR].iloc[0]
    rho_value = float(df_s[COL_RHO].iloc[0])

    w_method = canonical_w(w_raw)

    # --------------------------------------------------------
    # ESF sesuai W scenario
    # --------------------------------------------------------

    E = ESF_GRID10[w_method]

    eigenvalues = (
        EIGVAL_GRID10[w_method]
    )

    selected_ev = pilih_ev_moran(
        eigenvalues,
        threshold=MORAN_THRESHOLD
    )

    n_ev = len(selected_ev)

    print(
        f"\n[{nomor:02d}/18] "
        f"{scenario_id} | "
        f"EV Moran = {n_ev}"
    )

    # --------------------------------------------------------
    # OOF
    # --------------------------------------------------------

    oof_pred = np.full(
        len(y),
        np.nan
    )

    best_hyper_fold = []
    inner_rmse_fold = []

    # ========================================================
    # 5 OUTER FOLD
    # ========================================================

    for outer_fold in range(OUTER_FOLDS):

        test_idx = np.where(
            OUTER_LABELS_GRID10 == outer_fold
        )[0]

        train_idx = np.where(
            OUTER_LABELS_GRID10 != outer_fold
        )[0]

        # ----------------------------------------------------
        # Base predictor + EV Moran
        # ----------------------------------------------------

        X_train = np.column_stack([
            X_base[train_idx],
            E[train_idx][:, selected_ev]
        ])

        X_test = np.column_stack([
            X_base[test_idx],
            E[test_idx][:, selected_ev]
        ])

        y_train = y[train_idx]
        y_test = y[test_idx]

        coords_train = coords[train_idx]

        # ----------------------------------------------------
        # Inner spatial CV
        # ----------------------------------------------------

        inner_splits = buat_inner_split(
            coords_train,
            n_splits=INNER_FOLDS,
            seed=SEED_MODEL + outer_fold
        )

        # ----------------------------------------------------
        # H01-H18
        # ----------------------------------------------------

        best_hp, tuning_table = (
            cari_hyperparameter_terbaik(
                X_train,
                y_train,
                inner_splits
            )
        )

        # ----------------------------------------------------
        # Fit best
        # ----------------------------------------------------

        model = buat_model_xgb(
            best_hp
        )

        model.fit(
            X_train,
            y_train
        )

        pred_test = model.predict(
            X_test
        )

        oof_pred[test_idx] = pred_test

        best_hyper_fold.append(
            best_hp["Hyper_ID"]
        )

        inner_rmse_fold.append(
            best_hp["Inner_CV_RMSE"]
        )

        # ----------------------------------------------------
        # Fold result
        # ----------------------------------------------------

        MORAN_FOLD_RESULTS.append({

            "Scenario_ID":
                scenario_id,

            "Grid_Size":
                10,

            "Pattern":
                pattern,

            "W_Method":
                w_method,

            "Error_Structure":
                error_type,

            "Rho":
                rho_value,

            "Model":
                "ESF-XGBoost Moran",

            "Outer_Fold":
                outer_fold + 1,

            "Best_Hyper_ID":
                best_hp["Hyper_ID"],

            "n_estimators":
                int(best_hp["n_estimators"]),

            "learning_rate":
                float(best_hp["learning_rate"]),

            "max_depth":
                int(best_hp["max_depth"]),

            "N_EV_Selected":
                n_ev,

            "Inner_CV_RMSE":
                float(
                    best_hp["Inner_CV_RMSE"]
                ),

            "R2_Test":
                float(
                    r2_score(
                        y_test,
                        pred_test
                    )
                ),

            "RMSE_Test":
                float(
                    hitung_rmse(
                        y_test,
                        pred_test
                    )
                ),

            "MAE_Test":
                float(
                    mean_absolute_error(
                        y_test,
                        pred_test
                    )
                )
        })

        print(
            f"   Fold {outer_fold + 1}"
            f" → {best_hp['Hyper_ID']}"
        )

    # ========================================================
    # OOF VALIDATION
    # ========================================================

    assert np.isfinite(
        oof_pred
    ).all()

    R2_OOF = float(
        r2_score(
            y,
            oof_pred
        )
    )

    RMSE_OOF = float(
        hitung_rmse(
            y,
            oof_pred
        )
    )

    MAE_OOF = float(
        mean_absolute_error(
            y,
            oof_pred
        )
    )

    # ========================================================
    # HYPERPARAMETER DOMINAN
    # ========================================================

    counter = Counter(
        best_hyper_fold
    )

    max_freq = max(
        counter.values()
    )

    kandidat = [
        h
        for h, f in counter.items()
        if f == max_freq
    ]

    # Kalau seri, ambil ID terkecil dulu.
    # Detail RMSE per fold tetap tersimpan.
    dominant_hyper = sorted(
        kandidat
    )[0]

    hp_dom = (
        HYPERPARAMETERS[
            HYPERPARAMETERS["Hyper_ID"]
            == dominant_hyper
        ]
        .iloc[0]
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    MORAN_SUMMARY.append({

        "Scenario_ID":
            scenario_id,

        "Grid_Size":
            10,

        "Pattern":
            pattern,

        "W_Method":
            w_method,

        "Error_Structure":
            error_type,

        "Rho":
            rho_value,

        "Model":
            "ESF-XGBoost Moran",

        "ESF_Selection_Method":
            "Moran",

        "Best_Hyper_Fold1":
            best_hyper_fold[0],

        "Best_Hyper_Fold2":
            best_hyper_fold[1],

        "Best_Hyper_Fold3":
            best_hyper_fold[2],

        "Best_Hyper_Fold4":
            best_hyper_fold[3],

        "Best_Hyper_Fold5":
            best_hyper_fold[4],

        "Dominant_Hyper_ID":
            dominant_hyper,

        "Hyper_Frequency":
            f"{max_freq}/5",

        "n_estimators":
            int(hp_dom["n_estimators"]),

        "learning_rate":
            float(hp_dom["learning_rate"]),

        "max_depth":
            int(hp_dom["max_depth"]),

        "N_EV_Fold1":
            n_ev,

        "N_EV_Fold2":
            n_ev,

        "N_EV_Fold3":
            n_ev,

        "N_EV_Fold4":
            n_ev,

        "N_EV_Fold5":
            n_ev,

        "Mean_N_EV_Selected":
            float(n_ev),

        "Mean_Inner_CV_RMSE":
            float(
                np.mean(
                    inner_rmse_fold
                )
            ),

        "R2_OOF":
            R2_OOF,

        "RMSE_OOF":
            RMSE_OOF,

        "MAE_OOF":
            MAE_OOF,

        "Status":
            "OK"
    })

    # ========================================================
    # DETAIL PREDICTION
    # ========================================================

    MORAN_PREDICTIONS[
        scenario_id
    ] = pd.DataFrame({

        "Scenario_ID":
            scenario_id,

        "Location_ID":
            (
                df_s[COL_LOCATION].to_numpy()
                if COL_LOCATION is not None
                else np.arange(1, 101)
            ),

        "Y_Actual":
            y,

        "Pred_ESF_Moran":
            oof_pred,

        "Residual_ESF_Moran":
            y - oof_pred
    })

    print(
        f"   ✅ R2={R2_OOF:.4f} | "
        f"RMSE={RMSE_OOF:.4f} | "
        f"MAE={MAE_OOF:.4f}"
    )


# ============================================================
# DATAFRAME
# ============================================================

MORAN_SUMMARY_GRID10 = pd.DataFrame(
    MORAN_SUMMARY
)

MORAN_FOLD_GRID10 = pd.DataFrame(
    MORAN_FOLD_RESULTS
)

print("\n🎉 ESF-XGBOOST MORAN GRID10 SELESAI")


[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS | EV Moran = 26
   Fold 1 → H16
   Fold 2 → H07
   Fold 3 → H16
   Fold 4 → H16
   Fold 5 → H14
   ✅ R2=0.2525 | RMSE=2.7730 | MAE=2.0518

[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS | EV Moran = 26
   Fold 1 → H16
   Fold 2 → H13
   Fold 3 → H16
   Fold 4 → H08
   Fold 5 → H14
   ✅ R2=0.2218 | RMSE=2.7967 | MAE=2.0710

[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS | EV Moran = 21
   Fold 1 → H10
   Fold 2 → H08
   Fold 3 → H13
   Fold 4 → H13
   Fold 5 → H16
   ✅ R2=0.1868 | RMSE=2.8885 | MAE=2.1778

[04/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS | EV Moran = 21
   Fold 1 → H13
   Fold 2 → H14
   Fold 3 → H16
   Fold 4 → H14
   Fold 5 → H10
   ✅ R2=0.1955 | RMSE=2.8397 | MAE=2.1056

[05/18] G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS | EV Moran = 12
   Fold 1 → H17
   Fold 2 → H05
   Fold 3 → H14
   Fold 4 → H15
   Fold 5 → H14
   ✅ R2=0.3085 | RMSE=2.6042 | MAE=1.9491

[06/18] G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS | EV Moran = 12
   Fold 1 → H1

In [36]:
# ============================================================
# VALIDASI ESF-XGBOOST MORAN
# ============================================================

assert len(MORAN_SUMMARY_GRID10) == 18
assert len(MORAN_FOLD_GRID10) == 90
assert len(MORAN_PREDICTIONS) == 18

assert MORAN_SUMMARY_GRID10[
    "R2_OOF"
].notna().all()

assert MORAN_SUMMARY_GRID10[
    "RMSE_OOF"
].notna().all()

print("=" * 65)
print("✅ ESF-XGBOOST MORAN GRID10: PASS")
print("=" * 65)

display(
    MORAN_SUMMARY_GRID10[
        [
            "Scenario_ID",
            "W_Method",
            "N_EV_Fold1",
            "Best_Hyper_Fold1",
            "Best_Hyper_Fold2",
            "Best_Hyper_Fold3",
            "Best_Hyper_Fold4",
            "Best_Hyper_Fold5",
            "Dominant_Hyper_ID",
            "R2_OOF",
            "RMSE_OOF",
            "MAE_OOF"
        ]
    ]
)

✅ ESF-XGBOOST MORAN GRID10: PASS


,Scenario_ID,W_Method,N_EV_Fold1,Best_Hyper_Fold1,Best_Hyper_Fold2,Best_Hyper_Fold3,Best_Hyper_Fold4,Best_Hyper_Fold5,Dominant_Hyper_ID,R2_OOF,RMSE_OOF,MAE_OOF
0,G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS,KNN5,26,H16,H07,H16,H16,H14,H16,0.252504,2.773002,2.051804
1,G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS,KNN5,26,H16,H13,H16,H08,H14,H16,0.221824,2.796668,2.070961
2,G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,21,H10,H08,H13,H13,H16,H13,0.186826,2.888479,2.177849
3,G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,21,H13,H14,H16,H14,H10,H14,0.195462,2.839719,2.105608
4,G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS,IDW,12,H17,H05,H14,H15,H14,H14,0.308526,2.604169,1.949066
5,G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS,IDW,12,H13,H05,H04,H08,H11,H04,0.266620,2.649467,1.973914
6,G10_KORIDOR_KNN5_WEAK_HOMOGENEOUS,KNN5,26,H11,H01,H04,H10,H15,H01,0.475915,2.174960,1.728336
7,G10_KORIDOR_KNN5_WEAK_HETEROGENEOUS,KNN5,26,H12,H02,H10,H10,H12,H10,0.460840,2.185168,1.752939
8,G10_KORIDOR_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,21,H15,H11,H02,H08,H16,H02,0.451142,2.226963,1.759841
9,G10_KORIDOR_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,21,H12,H11,H02,H07,H17,H02,0.472741,2.161863,1.762001


## ESF-XGBoost LASSO

In [37]:
# ============================================================
# ESF-XGBOOST LASSO
# WEAK - GRID10
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from collections import Counter
import warnings

LASSO_SUMMARY = []
LASSO_FOLD_RESULTS = []
LASSO_PREDICTIONS = {}


# ============================================================
# CROSS-FITTED BASELINE RESIDUAL
# outer-training saja
# ============================================================

def baseline_crossfit_residual(
    X_train,
    y_train,
    coords_train,
    seed
):

    inner_splits = buat_inner_split(
        coords_train,
        n_splits=INNER_FOLDS,
        seed=seed
    )

    # Cari hyperparameter baseline terbaik
    best_hp, _ = cari_hyperparameter_terbaik(
        X_train,
        y_train,
        inner_splits
    )

    cf_pred = np.full(
        len(y_train),
        np.nan
    )

    for tr_idx, val_idx in inner_splits:

        model = buat_model_xgb(
            best_hp
        )

        model.fit(
            X_train[tr_idx],
            y_train[tr_idx]
        )

        cf_pred[val_idx] = model.predict(
            X_train[val_idx]
        )

    assert np.isfinite(cf_pred).all()

    residual = (
        y_train - cf_pred
    )

    return residual


# ============================================================
# SELEKSI EV DENGAN LASSO
# menggunakan seluruh EV
# ============================================================

def pilih_ev_lasso(
    E_train,
    residual_train,
    seed
):

    scaler = StandardScaler()

    E_scaled = scaler.fit_transform(
        E_train
    )

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore"
        )

        lasso = LassoCV(
            alphas=np.logspace(
                -4,
                0,
                60
            ),
            cv=3,
            random_state=seed,
            max_iter=30000,
            n_jobs=-1
        )

        lasso.fit(
            E_scaled,
            residual_train
        )

    selected = np.where(
        np.abs(lasso.coef_) > 1e-8
    )[0]

    return selected


# ============================================================
# LOOP 18 SCENARIO
# ============================================================

for nomor, scenario_id in enumerate(
    SCENARIO_IDS,
    start=1
):

    df_s = (
        DATA_MODEL[
            DATA_MODEL[COL_SCENARIO] == scenario_id
        ]
        .copy()
        .sort_values(
            [COL_YCOORD, COL_XCOORD]
        )
        .reset_index(drop=True)
    )

    X_base = (
        df_s[PREDICTOR_COLS]
        .to_numpy(dtype=float)
    )

    y = (
        df_s[COL_TARGET]
        .to_numpy(dtype=float)
    )

    coords = (
        df_s[
            [COL_XCOORD, COL_YCOORD]
        ]
        .to_numpy(dtype=float)
    )

    pattern = df_s[COL_PATTERN].iloc[0]
    w_raw = df_s[COL_W].iloc[0]
    error_type = df_s[COL_ERROR].iloc[0]
    rho_value = float(df_s[COL_RHO].iloc[0])

    w_method = canonical_w(
        w_raw
    )

    E = ESF_GRID10[
        w_method
    ]

    # --------------------------------------------------------
    # Storage scenario
    # --------------------------------------------------------

    oof_pred = np.full(
        len(y),
        np.nan
    )

    best_hyper_fold = []
    inner_rmse_fold = []
    n_ev_fold = []

    print(
        f"\n[{nomor:02d}/18] "
        f"{scenario_id}"
    )


    # ========================================================
    # 5 OUTER FOLD
    # ========================================================

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_idx = np.where(
            OUTER_LABELS_GRID10 == outer_fold
        )[0]

        train_idx = np.where(
            OUTER_LABELS_GRID10 != outer_fold
        )[0]


        X_train_base = X_base[
            train_idx
        ]

        X_test_base = X_base[
            test_idx
        ]

        y_train = y[
            train_idx
        ]

        y_test = y[
            test_idx
        ]

        coords_train = coords[
            train_idx
        ]

        E_train = E[
            train_idx,
            :
        ]

        E_test = E[
            test_idx,
            :
        ]


        # ====================================================
        # 1. BASELINE RESIDUAL
        # hanya dari outer-training
        # ====================================================

        residual_train = (
            baseline_crossfit_residual(
                X_train_base,
                y_train,
                coords_train,
                seed=
                    SEED_MODEL
                    + 100
                    + outer_fold
            )
        )


        # ====================================================
        # 2. LASSO PILIH EV
        # ====================================================

        selected_ev = pilih_ev_lasso(
            E_train,
            residual_train,
            seed=
                SEED_MODEL
                + outer_fold
        )

        n_ev = len(
            selected_ev
        )

        n_ev_fold.append(
            n_ev
        )


        # ====================================================
        # 3. X + EV TERPILIH
        # ====================================================

        if n_ev > 0:

            X_train = np.column_stack([
                X_train_base,
                E_train[
                    :,
                    selected_ev
                ]
            ])

            X_test = np.column_stack([
                X_test_base,
                E_test[
                    :,
                    selected_ev
                ]
            ])

        else:

            X_train = (
                X_train_base.copy()
            )

            X_test = (
                X_test_base.copy()
            )


        # ====================================================
        # 4. INNER SPATIAL CV H01-H18
        # ====================================================

        inner_splits = buat_inner_split(
            coords_train,
            n_splits=INNER_FOLDS,
            seed=
                SEED_MODEL
                + outer_fold
        )

        best_hp, tuning_table = (
            cari_hyperparameter_terbaik(
                X_train,
                y_train,
                inner_splits
            )
        )


        # ====================================================
        # 5. FIT BEST XGBOOST
        # ====================================================

        model = buat_model_xgb(
            best_hp
        )

        model.fit(
            X_train,
            y_train
        )

        pred_test = model.predict(
            X_test
        )

        oof_pred[
            test_idx
        ] = pred_test


        best_hyper_fold.append(
            best_hp["Hyper_ID"]
        )

        inner_rmse_fold.append(
            best_hp[
                "Inner_CV_RMSE"
            ]
        )


        # ====================================================
        # FOLD RESULT
        # ====================================================

        LASSO_FOLD_RESULTS.append({

            "Scenario_ID":
                scenario_id,

            "Grid_Size":
                10,

            "Pattern":
                pattern,

            "W_Method":
                w_method,

            "Error_Structure":
                error_type,

            "Rho":
                rho_value,

            "Model":
                "ESF-XGBoost LASSO",

            "Outer_Fold":
                outer_fold + 1,

            "Best_Hyper_ID":
                best_hp["Hyper_ID"],

            "n_estimators":
                int(
                    best_hp[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    best_hp[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    best_hp[
                        "max_depth"
                    ]
                ),

            "N_EV_Selected":
                n_ev,

            "Inner_CV_RMSE":
                float(
                    best_hp[
                        "Inner_CV_RMSE"
                    ]
                ),

            "R2_Test":
                float(
                    r2_score(
                        y_test,
                        pred_test
                    )
                ),

            "RMSE_Test":
                float(
                    hitung_rmse(
                        y_test,
                        pred_test
                    )
                ),

            "MAE_Test":
                float(
                    mean_absolute_error(
                        y_test,
                        pred_test
                    )
                )
        })


        print(
            f"   Fold {outer_fold + 1}"
            f" → EV={n_ev:2d}"
            f" | Best={best_hp['Hyper_ID']}"
        )


    # ========================================================
    # OOF EVALUATION
    # ========================================================

    assert np.isfinite(
        oof_pred
    ).all()


    R2_OOF = float(
        r2_score(
            y,
            oof_pred
        )
    )

    RMSE_OOF = float(
        hitung_rmse(
            y,
            oof_pred
        )
    )

    MAE_OOF = float(
        mean_absolute_error(
            y,
            oof_pred
        )
    )


    # ========================================================
    # HYPERPARAMETER DOMINAN
    # ========================================================

    counter = Counter(
        best_hyper_fold
    )

    max_freq = max(
        counter.values()
    )

    kandidat = [
        h
        for h, f in counter.items()
        if f == max_freq
    ]

    dominant_hyper = sorted(
        kandidat
    )[0]


    hp_dom = (
        HYPERPARAMETERS[
            HYPERPARAMETERS[
                "Hyper_ID"
            ] == dominant_hyper
        ]
        .iloc[0]
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    LASSO_SUMMARY.append({

        "Scenario_ID":
            scenario_id,

        "Grid_Size":
            10,

        "Pattern":
            pattern,

        "W_Method":
            w_method,

        "Error_Structure":
            error_type,

        "Rho":
            rho_value,

        "Model":
            "ESF-XGBoost LASSO",

        "ESF_Selection_Method":
            "LASSO",

        "Best_Hyper_Fold1":
            best_hyper_fold[0],

        "Best_Hyper_Fold2":
            best_hyper_fold[1],

        "Best_Hyper_Fold3":
            best_hyper_fold[2],

        "Best_Hyper_Fold4":
            best_hyper_fold[3],

        "Best_Hyper_Fold5":
            best_hyper_fold[4],

        "Dominant_Hyper_ID":
            dominant_hyper,

        "Hyper_Frequency":
            f"{max_freq}/5",

        "n_estimators":
            int(
                hp_dom[
                    "n_estimators"
                ]
            ),

        "learning_rate":
            float(
                hp_dom[
                    "learning_rate"
                ]
            ),

        "max_depth":
            int(
                hp_dom[
                    "max_depth"
                ]
            ),

        "N_EV_Fold1":
            n_ev_fold[0],

        "N_EV_Fold2":
            n_ev_fold[1],

        "N_EV_Fold3":
            n_ev_fold[2],

        "N_EV_Fold4":
            n_ev_fold[3],

        "N_EV_Fold5":
            n_ev_fold[4],

        "Mean_N_EV_Selected":
            float(
                np.mean(
                    n_ev_fold
                )
            ),

        "Mean_Inner_CV_RMSE":
            float(
                np.mean(
                    inner_rmse_fold
                )
            ),

        "R2_OOF":
            R2_OOF,

        "RMSE_OOF":
            RMSE_OOF,

        "MAE_OOF":
            MAE_OOF,

        "Status":
            "OK"
    })


    # ========================================================
    # DETAIL PREDICTION
    # ========================================================

    LASSO_PREDICTIONS[
        scenario_id
    ] = pd.DataFrame({

        "Scenario_ID":
            scenario_id,

        "Location_ID":
            (
                df_s[
                    COL_LOCATION
                ].to_numpy()

                if COL_LOCATION
                is not None

                else
                np.arange(
                    1,
                    101
                )
            ),

        "Y_Actual":
            y,

        "Pred_ESF_LASSO":
            oof_pred,

        "Residual_ESF_LASSO":
            y - oof_pred
    })


    print(
        f"   ✅ OOF "
        f"R2={R2_OOF:.4f} | "
        f"RMSE={RMSE_OOF:.4f} | "
        f"MAE={MAE_OOF:.4f} | "
        f"Mean EV="
        f"{np.mean(n_ev_fold):.1f}"
    )


# ============================================================
# DATAFRAME FINAL
# ============================================================

LASSO_SUMMARY_GRID10 = pd.DataFrame(
    LASSO_SUMMARY
)

LASSO_FOLD_GRID10 = pd.DataFrame(
    LASSO_FOLD_RESULTS
)

print("\n" + "=" * 70)
print("🎉 ESF-XGBOOST LASSO GRID10 SELESAI")
print("=" * 70)


[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS
   Fold 1 → EV=15 | Best=H10
   Fold 2 → EV= 3 | Best=H08
   Fold 3 → EV= 0 | Best=H13
   Fold 4 → EV=18 | Best=H10
   Fold 5 → EV= 0 | Best=H04
   ✅ OOF R2=0.2060 | RMSE=2.8580 | MAE=2.1319 | Mean EV=7.2

[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS
   Fold 1 → EV=15 | Best=H10
   Fold 2 → EV= 3 | Best=H04
   Fold 3 → EV= 0 | Best=H14
   Fold 4 → EV=26 | Best=H10
   Fold 5 → EV= 0 | Best=H04
   ✅ OOF R2=0.1972 | RMSE=2.8405 | MAE=2.1005 | Mean EV=8.8

[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS
   Fold 1 → EV= 0 | Best=H10
   Fold 2 → EV= 0 | Best=H12
   Fold 3 → EV= 0 | Best=H10
   Fold 4 → EV= 4 | Best=H10
   Fold 5 → EV= 0 | Best=H10
   ✅ OOF R2=0.3218 | RMSE=2.6378 | MAE=1.9490 | Mean EV=0.8

[04/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS
   Fold 1 → EV= 0 | Best=H10
   Fold 2 → EV= 0 | Best=H06
   Fold 3 → EV= 0 | Best=H17
   Fold 4 → EV= 2 | Best=H13
   Fold 5 → EV= 0 | Best=H10
   ✅ OOF R2=0.2943 | RMSE=2.6596 | MAE=1.9995 

In [38]:
# ============================================================
# VALIDASI ESF-XGBOOST LASSO
# ============================================================

assert len(
    LASSO_SUMMARY_GRID10
) == 18

assert len(
    LASSO_FOLD_GRID10
) == 90

assert len(
    LASSO_PREDICTIONS
) == 18

assert (
    LASSO_SUMMARY_GRID10[
        "R2_OOF"
    ]
    .notna()
    .all()
)

assert (
    LASSO_SUMMARY_GRID10[
        "RMSE_OOF"
    ]
    .notna()
    .all()
)

assert (
    LASSO_SUMMARY_GRID10[
        "MAE_OOF"
    ]
    .notna()
    .all()
)


print("=" * 65)
print("✅ ESF-XGBOOST LASSO GRID10: PASS")
print("=" * 65)

display(
    LASSO_SUMMARY_GRID10[
        [
            "Scenario_ID",
            "W_Method",

            "N_EV_Fold1",
            "N_EV_Fold2",
            "N_EV_Fold3",
            "N_EV_Fold4",
            "N_EV_Fold5",

            "Mean_N_EV_Selected",

            "Dominant_Hyper_ID",

            "R2_OOF",
            "RMSE_OOF",
            "MAE_OOF"
        ]
    ]
)

✅ ESF-XGBOOST LASSO GRID10: PASS


,Scenario_ID,W_Method,N_EV_Fold1,N_EV_Fold2,N_EV_Fold3,N_EV_Fold4,N_EV_Fold5,Mean_N_EV_Selected,Dominant_Hyper_ID,R2_OOF,RMSE_OOF,MAE_OOF
0,G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS,KNN5,15,3,0,18,0,7.2,H10,0.205993,2.857972,2.131950
1,G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS,KNN5,15,3,0,26,0,8.8,H04,0.197229,2.840521,2.100467
2,G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,0,0,0,4,0,0.8,H10,0.321846,2.637799,1.949036
3,G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,0,0,0,2,0,0.4,H10,0.294300,2.659574,1.999455
4,G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS,IDW,0,0,0,1,0,0.2,H10,0.306677,2.607648,1.950754
5,G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS,IDW,0,0,0,1,0,0.2,H04,0.301790,2.585157,1.959988
6,G10_KORIDOR_KNN5_WEAK_HOMOGENEOUS,KNN5,0,1,0,0,0,0.2,H01,0.503827,2.116250,1.667516
7,G10_KORIDOR_KNN5_WEAK_HETEROGENEOUS,KNN5,0,1,0,4,0,1.0,H04,0.500556,2.103146,1.702091
8,G10_KORIDOR_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,0,0,0,0,0,0.0,H04,0.527318,2.066652,1.641310
9,G10_KORIDOR_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,0,0,0,0,0,0.0,H05,0.507843,2.088662,1.721827


## ESF-XGBoost Stepwise

In [48]:
# ============================================================
# STEPWISE FINAL
# SEMUA 99 EV
# FORWARD SELECTION BERDASARKAN SPATIAL CV-RMSE
# TANPA BIC / EBIC
# ============================================================

import numpy as np
import pandas as pd

from collections import Counter
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


# ============================================================
# STORAGE BARU
# ============================================================

STEPWISE_SUMMARY = []
STEPWISE_FOLD_RESULTS = []
STEPWISE_PREDICTIONS = {}


# ============================================================
# CV-RMSE UNTUK RESIDUAL
# ============================================================

def cv_rmse_stepwise_residual(
    E_train,
    residual_train,
    selected_cols,
    inner_splits
):
    """
    Mengukur kemampuan EV menjelaskan residual baseline
    dengan inner spatial CV.
    """

    fold_rmse = []

    for tr_idx, val_idx in inner_splits:

        y_tr = residual_train[tr_idx]
        y_val = residual_train[val_idx]

        # ----------------------------------------------------
        # Belum ada EV:
        # model hanya menggunakan mean residual training
        # ----------------------------------------------------

        if len(selected_cols) == 0:

            pred_val = np.full(
                len(val_idx),
                np.mean(y_tr)
            )

        else:

            X_tr = E_train[
                tr_idx
            ][:, selected_cols]

            X_val = E_train[
                val_idx
            ][:, selected_cols]

            model = LinearRegression()

            model.fit(
                X_tr,
                y_tr
            )

            pred_val = model.predict(
                X_val
            )

        rmse_val = np.sqrt(
            mean_squared_error(
                y_val,
                pred_val
            )
        )

        fold_rmse.append(
            rmse_val
        )

    return float(
        np.mean(
            fold_rmse
        )
    )


# ============================================================
# FORWARD STEPWISE
# SEMUA 99 EV BOLEH DIPILIH
# ============================================================

def pilih_ev_stepwise_spatial_cv(
    E_train,
    residual_train,
    inner_splits
):

    p_total = E_train.shape[1]

    # Grid10 = EV 0 sampai EV 98
    remaining = list(
        range(p_total)
    )

    selected = []

    # --------------------------------------------------------
    # RMSE awal tanpa EV
    # --------------------------------------------------------

    current_rmse = (
        cv_rmse_stepwise_residual(
            E_train,
            residual_train,
            selected_cols=[],
            inner_splits=inner_splits
        )
    )

    selection_history = []

    # ========================================================
    # FORWARD SELECTION
    # ========================================================

    while len(remaining) > 0:

        candidate_results = []

        # ----------------------------------------------------
        # Coba semua EV yang belum terpilih
        # ----------------------------------------------------

        for candidate in remaining:

            trial_cols = (
                selected
                +
                [candidate]
            )

            candidate_rmse = (
                cv_rmse_stepwise_residual(
                    E_train,
                    residual_train,
                    selected_cols=
                        trial_cols,
                    inner_splits=
                        inner_splits
                )
            )

            candidate_results.append(
                (
                    candidate_rmse,
                    candidate
                )
            )

        # ----------------------------------------------------
        # Candidate dengan CV-RMSE terkecil
        # ----------------------------------------------------

        best_rmse, best_candidate = min(
            candidate_results,
            key=lambda x: x[0]
        )

        improvement = (
            current_rmse
            -
            best_rmse
        )

        # ----------------------------------------------------
        # Tambahkan hanya jika RMSE benar-benar membaik
        # ----------------------------------------------------

        if improvement > 1e-10:

            selected.append(
                best_candidate
            )

            remaining.remove(
                best_candidate
            )

            selection_history.append({
                "Step":
                    len(selected),

                "EV_Index":
                    best_candidate,

                "CV_RMSE_Before":
                    current_rmse,

                "CV_RMSE_After":
                    best_rmse,

                "Improvement":
                    improvement
            })

            current_rmse = (
                best_rmse
            )

        else:

            # Tidak ada EV lagi yang memperbaiki CV
            break

    return (
        np.asarray(
            selected,
            dtype=int
        ),
        float(current_rmse),
        selection_history
    )


# ============================================================
# HYPERPARAMETER DOMINAN
# jika seri -> mean inner CV RMSE terkecil
# ============================================================

def dominant_hyper_stepwise(
    hyper_ids,
    inner_rmse_values
):

    counts = Counter(
        hyper_ids
    )

    max_freq = max(
        counts.values()
    )

    candidates = [
        h
        for h, freq in counts.items()
        if freq == max_freq
    ]

    if len(candidates) == 1:

        dominant = candidates[0]

    else:

        mean_rmse = {}

        for h in candidates:

            values = [
                inner_rmse_values[i]
                for i in range(
                    len(hyper_ids)
                )
                if hyper_ids[i] == h
            ]

            mean_rmse[h] = np.mean(
                values
            )

        dominant = min(
            candidates,
            key=lambda h:
                mean_rmse[h]
        )

    return dominant, max_freq


print("✅ HELPER STEPWISE SPATIAL-CV SIAP")
print("✅ Candidate awal = semua 99 EV")
print("✅ Tidak menggunakan BIC / EBIC")


# ============================================================
# ESF-XGBOOST STEPWISE
# WEAK - GRID10
# ============================================================

for nomor, scenario_id in enumerate(
    SCENARIO_IDS,
    start=1
):

    # ========================================================
    # DATA SCENARIO
    # ========================================================

    df_s = (
        DATA_MODEL[
            DATA_MODEL[
                COL_SCENARIO
            ] == scenario_id
        ]
        .copy()
        .sort_values(
            [
                COL_YCOORD,
                COL_XCOORD
            ]
        )
        .reset_index(drop=True)
    )

    assert len(df_s) == 100


    # ========================================================
    # DATA
    # ========================================================

    X_base = (
        df_s[
            PREDICTOR_COLS
        ]
        .to_numpy(
            dtype=float
        )
    )

    y = (
        df_s[
            COL_TARGET
        ]
        .to_numpy(
            dtype=float
        )
    )

    coords = (
        df_s[
            [
                COL_XCOORD,
                COL_YCOORD
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )


    # ========================================================
    # METADATA
    # ========================================================

    pattern = (
        df_s[
            COL_PATTERN
        ].iloc[0]
    )

    w_raw = (
        df_s[
            COL_W
        ].iloc[0]
    )

    error_type = (
        df_s[
            COL_ERROR
        ].iloc[0]
    )

    rho_value = float(
        df_s[
            COL_RHO
        ].iloc[0]
    )

    w_method = canonical_w(
        w_raw
    )


    # ========================================================
    # SEMUA 99 EV
    # ========================================================

    E = ESF_GRID10[
        w_method
    ]

    assert E.shape == (
        100,
        99
    )


    print(
        f"\n[{nomor:02d}/18] "
        f"{scenario_id}"
    )

    print(
        "   Candidate EV = 99"
    )


    # ========================================================
    # STORAGE SCENARIO
    # ========================================================

    oof_pred = np.full(
        100,
        np.nan
    )

    best_hyper_fold = []

    inner_rmse_fold = []

    n_ev_fold = []

    selection_rmse_fold = []


    # ========================================================
    # 5 OUTER SPATIAL FOLD
    # ========================================================

    for outer_fold in range(
        OUTER_FOLDS
    ):

        # ----------------------------------------------------
        # Outer train/test
        # ----------------------------------------------------

        test_idx = np.where(
            OUTER_LABELS_GRID10
            ==
            outer_fold
        )[0]

        train_idx = np.where(
            OUTER_LABELS_GRID10
            !=
            outer_fold
        )[0]


        X_train_base = (
            X_base[
                train_idx
            ]
        )

        X_test_base = (
            X_base[
                test_idx
            ]
        )

        y_train = (
            y[
                train_idx
            ]
        )

        y_test = (
            y[
                test_idx
            ]
        )

        coords_train = (
            coords[
                train_idx
            ]
        )

        E_train = (
            E[
                train_idx,
                :
            ]
        )

        E_test = (
            E[
                test_idx,
                :
            ]
        )


        # ====================================================
        # INNER SPATIAL FOLD
        # ====================================================

        inner_splits = (
            buat_inner_split(
                coords_train,
                n_splits=
                    INNER_FOLDS,
                seed=
                    SEED_MODEL
                    +
                    outer_fold
            )
        )


        # ====================================================
        # 1. CROSS-FITTED BASELINE RESIDUAL
        #
        # Hanya outer-training
        # ====================================================

        residual_train = (
            baseline_crossfit_residual(
                X_train_base,
                y_train,
                coords_train,
                seed=
                    SEED_MODEL
                    +
                    400
                    +
                    outer_fold
            )
        )


        # ====================================================
        # 2. STEPWISE SPATIAL-CV
        #
        # SEMUA 99 EV menjadi kandidat
        # ====================================================

        (
            selected_ev,
            selection_cv_rmse,
            selection_history
        ) = pilih_ev_stepwise_spatial_cv(
            E_train,
            residual_train,
            inner_splits
        )


        n_ev = int(
            len(
                selected_ev
            )
        )


        n_ev_fold.append(
            n_ev
        )

        selection_rmse_fold.append(
            selection_cv_rmse
        )


        # ====================================================
        # 3. BASE X + EV TERPILIH
        # ====================================================

        if n_ev > 0:

            X_train = np.column_stack([
                X_train_base,

                E_train[
                    :,
                    selected_ev
                ]
            ])

            X_test = np.column_stack([
                X_test_base,

                E_test[
                    :,
                    selected_ev
                ]
            ])

        else:

            X_train = (
                X_train_base.copy()
            )

            X_test = (
                X_test_base.copy()
            )


        # ====================================================
        # 4. HYPERPARAMETER H01-H18
        #
        # Stepwise-XGBoost memilih sendiri
        # ====================================================

        best_hp, tuning_table = (
            cari_hyperparameter_terbaik(
                X_train,
                y_train,
                inner_splits
            )
        )


        best_hyper_fold.append(
            best_hp[
                "Hyper_ID"
            ]
        )

        inner_rmse_fold.append(
            float(
                best_hp[
                    "Inner_CV_RMSE"
                ]
            )
        )


        # ====================================================
        # 5. FIT BEST XGBOOST
        # ====================================================

        model = buat_model_xgb(
            best_hp
        )

        model.fit(
            X_train,
            y_train
        )

        pred_test = model.predict(
            X_test
        )


        # ====================================================
        # 6. SIMPAN OOF
        # ====================================================

        oof_pred[
            test_idx
        ] = pred_test


        # ====================================================
        # 7. FOLD RESULT
        # ====================================================

        STEPWISE_FOLD_RESULTS.append({

            "Scenario_ID":
                scenario_id,

            "Grid_Size":
                10,

            "Pattern":
                pattern,

            "W_Method":
                w_method,

            "Error_Structure":
                error_type,

            "Rho":
                rho_value,

            "Model":
                "ESF-XGBoost Stepwise",

            "Outer_Fold":
                outer_fold + 1,

            # Semua EV tersedia
            "Candidate_EV":
                99,

            "N_EV_Selected":
                n_ev,

            "Stepwise_CV_RMSE":
                selection_cv_rmse,

            "Best_Hyper_ID":
                best_hp[
                    "Hyper_ID"
                ],

            "n_estimators":
                int(
                    best_hp[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    best_hp[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    best_hp[
                        "max_depth"
                    ]
                ),

            "Inner_CV_RMSE":
                float(
                    best_hp[
                        "Inner_CV_RMSE"
                    ]
                ),

            "R2_Test":
                float(
                    r2_score(
                        y_test,
                        pred_test
                    )
                ),

            "RMSE_Test":
                float(
                    hitung_rmse(
                        y_test,
                        pred_test
                    )
                ),

            "MAE_Test":
                float(
                    mean_absolute_error(
                        y_test,
                        pred_test
                    )
                )
        })


        print(
            f"   Fold {outer_fold + 1}"
            f" → EV={n_ev:2d}"
            f" | StepCV="
            f"{selection_cv_rmse:.4f}"
            f" | Best="
            f"{best_hp['Hyper_ID']}"
        )


    # ========================================================
    # OOF VALIDATION
    # ========================================================

    assert np.isfinite(
        oof_pred
    ).all()


    # ========================================================
    # OOF METRICS
    # ========================================================

    R2_OOF = float(
        r2_score(
            y,
            oof_pred
        )
    )

    RMSE_OOF = float(
        hitung_rmse(
            y,
            oof_pred
        )
    )

    MAE_OOF = float(
        mean_absolute_error(
            y,
            oof_pred
        )
    )


    # ========================================================
    # HYPERPARAMETER DOMINAN
    # ========================================================

    (
        dominant_hyper,
        max_freq
    ) = dominant_hyper_stepwise(
        best_hyper_fold,
        inner_rmse_fold
    )


    hp_dom = (
        HYPERPARAMETERS[
            HYPERPARAMETERS[
                "Hyper_ID"
            ] == dominant_hyper
        ]
        .iloc[0]
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    STEPWISE_SUMMARY.append({

        "Scenario_ID":
            scenario_id,

        "Grid_Size":
            10,

        "Pattern":
            pattern,

        "W_Method":
            w_method,

        "Error_Structure":
            error_type,

        "Rho":
            rho_value,

        "Model":
            "ESF-XGBoost Stepwise",

        "ESF_Selection_Method":
            "Forward Spatial-CV",

        "Candidate_EV":
            99,

        "Best_Hyper_Fold1":
            best_hyper_fold[0],

        "Best_Hyper_Fold2":
            best_hyper_fold[1],

        "Best_Hyper_Fold3":
            best_hyper_fold[2],

        "Best_Hyper_Fold4":
            best_hyper_fold[3],

        "Best_Hyper_Fold5":
            best_hyper_fold[4],

        "Dominant_Hyper_ID":
            dominant_hyper,

        "Hyper_Frequency":
            f"{max_freq}/5",

        "n_estimators":
            int(
                hp_dom[
                    "n_estimators"
                ]
            ),

        "learning_rate":
            float(
                hp_dom[
                    "learning_rate"
                ]
            ),

        "max_depth":
            int(
                hp_dom[
                    "max_depth"
                ]
            ),

        "N_EV_Fold1":
            n_ev_fold[0],

        "N_EV_Fold2":
            n_ev_fold[1],

        "N_EV_Fold3":
            n_ev_fold[2],

        "N_EV_Fold4":
            n_ev_fold[3],

        "N_EV_Fold5":
            n_ev_fold[4],

        "Mean_N_EV_Selected":
            float(
                np.mean(
                    n_ev_fold
                )
            ),

        "Mean_Stepwise_CV_RMSE":
            float(
                np.mean(
                    selection_rmse_fold
                )
            ),

        "Mean_Inner_CV_RMSE":
            float(
                np.mean(
                    inner_rmse_fold
                )
            ),

        "R2_OOF":
            R2_OOF,

        "RMSE_OOF":
            RMSE_OOF,

        "MAE_OOF":
            MAE_OOF,

        "Status":
            "OK"
    })


    # ========================================================
    # DETAIL PREDICTION
    # ========================================================

    STEPWISE_PREDICTIONS[
        scenario_id
    ] = pd.DataFrame({

        "Scenario_ID":
            scenario_id,

        "Location_ID":
            (
                df_s[
                    COL_LOCATION
                ].to_numpy()

                if COL_LOCATION
                is not None

                else
                np.arange(
                    1,
                    101
                )
            ),

        "Y_Actual":
            y,

        "Pred_ESF_Stepwise":
            oof_pred,

        "Residual_ESF_Stepwise":
            y - oof_pred
    })


    print(
        f"   ✅ OOF "
        f"R2={R2_OOF:.4f} | "
        f"RMSE={RMSE_OOF:.4f} | "
        f"MAE={MAE_OOF:.4f} | "
        f"Mean EV="
        f"{np.mean(n_ev_fold):.1f}"
    )


# ============================================================
# FINAL DATAFRAME
# ============================================================

STEPWISE_SUMMARY_GRID10 = pd.DataFrame(
    STEPWISE_SUMMARY
)

STEPWISE_FOLD_GRID10 = pd.DataFrame(
    STEPWISE_FOLD_RESULTS
)


print("\n" + "=" * 70)
print("🎉 ESF-XGBOOST STEPWISE GRID10 SELESAI")
print("=" * 70)

print(
    "Summary:",
    STEPWISE_SUMMARY_GRID10.shape
)

print(
    "Fold:",
    STEPWISE_FOLD_GRID10.shape
)

print(
    "Predictions:",
    len(
        STEPWISE_PREDICTIONS
    )
)

✅ HELPER STEPWISE SPATIAL-CV SIAP
✅ Candidate awal = semua 99 EV
✅ Tidak menggunakan BIC / EBIC

[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS
   Candidate EV = 99
   Fold 1 → EV=24 | StepCV=1.5289 | Best=H10
   Fold 2 → EV=27 | StepCV=1.2367 | Best=H17
   Fold 3 → EV=27 | StepCV=1.2536 | Best=H16
   Fold 4 → EV=20 | StepCV=1.3226 | Best=H10
   Fold 5 → EV=19 | StepCV=1.5345 | Best=H16
   ✅ OOF R2=0.2283 | RMSE=2.8175 | MAE=2.0720 | Mean EV=23.4

[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS
   Candidate EV = 99
   Fold 1 → EV=28 | StepCV=1.2724 | Best=H13
   Fold 2 → EV=27 | StepCV=1.3613 | Best=H10
   Fold 3 → EV=23 | StepCV=1.4455 | Best=H13
   Fold 4 → EV=24 | StepCV=1.4156 | Best=H13
   Fold 5 → EV=24 | StepCV=1.3099 | Best=H07
   ✅ OOF R2=0.1283 | RMSE=2.9600 | MAE=2.2250 | Mean EV=25.2

[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS
   Candidate EV = 99
   Fold 1 → EV=27 | StepCV=1.3981 | Best=H13
   Fold 2 → EV=26 | StepCV=1.5316 | Best=H17
   Fold 3 → EV=31 | StepCV=1.3774 | Bes

In [49]:
# ============================================================
# VALIDASI STEPWISE SPATIAL-CV
# ============================================================

assert len(
    STEPWISE_SUMMARY_GRID10
) == 18

assert len(
    STEPWISE_FOLD_GRID10
) == 90

assert len(
    STEPWISE_PREDICTIONS
) == 18


assert (
    STEPWISE_SUMMARY_GRID10[
        "Candidate_EV"
    ] == 99
).all()


assert (
    STEPWISE_SUMMARY_GRID10[
        "R2_OOF"
    ]
    .notna()
    .all()
)

assert (
    STEPWISE_SUMMARY_GRID10[
        "RMSE_OOF"
    ]
    .notna()
    .all()
)

assert (
    STEPWISE_SUMMARY_GRID10[
        "MAE_OOF"
    ]
    .notna()
    .all()
)


print("=" * 70)
print("✅ ESF-XGBOOST STEPWISE GRID10: PASS")
print("=" * 70)


display(
    STEPWISE_SUMMARY_GRID10[
        [
            "Scenario_ID",
            "W_Method",

            "Candidate_EV",

            "N_EV_Fold1",
            "N_EV_Fold2",
            "N_EV_Fold3",
            "N_EV_Fold4",
            "N_EV_Fold5",

            "Mean_N_EV_Selected",

            "Dominant_Hyper_ID",

            "R2_OOF",
            "RMSE_OOF",
            "MAE_OOF"
        ]
    ]
)

✅ ESF-XGBOOST STEPWISE GRID10: PASS


,Scenario_ID,W_Method,Candidate_EV,N_EV_Fold1,N_EV_Fold2,N_EV_Fold3,N_EV_Fold4,N_EV_Fold5,Mean_N_EV_Selected,Dominant_Hyper_ID,R2_OOF,RMSE_OOF,MAE_OOF
0,G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS,KNN5,99,24,27,27,20,19,23.4,H10,0.228327,2.817491,2.072014
1,G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS,KNN5,99,28,27,23,24,24,25.2,H13,0.128301,2.959957,2.224977
2,G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,99,27,26,31,25,14,24.6,H13,0.077455,3.076600,2.305016
3,G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,99,31,22,24,25,22,24.8,H13,0.151714,2.915904,2.212410
4,G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS,IDW,99,25,27,28,22,30,26.4,H13,0.127546,2.925179,2.221749
5,G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS,IDW,99,28,27,28,31,27,28.2,H13,0.164391,2.828104,2.166837
6,G10_KORIDOR_KNN5_WEAK_HOMOGENEOUS,KNN5,99,25,21,26,25,20,23.4,H10,0.435080,2.258104,1.759771
7,G10_KORIDOR_KNN5_WEAK_HETEROGENEOUS,KNN5,99,26,22,25,28,29,26.0,H13,0.387985,2.328128,1.801576
8,G10_KORIDOR_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,99,28,24,26,27,24,25.8,H10,0.389847,2.348022,1.883407
9,G10_KORIDOR_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,99,22,31,23,29,26,26.2,H13,0.348983,2.402217,1.968331


## ESF-XGBOOST SHAP


In [50]:
# ============================================================
# ESF-XGBOOST SHAP
# HELPER
# WEAK - GRID10
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb

from collections import Counter


# ============================================================
# SETTING SHAP
# ============================================================

SHAP_CUMULATIVE_THRESHOLD = 0.90


# ============================================================
# STORAGE BARU
# ============================================================

SHAP_SUMMARY = []
SHAP_FOLD_RESULTS = []
SHAP_PREDICTIONS = {}


# ============================================================
# HYPERPARAMETER DOMINAN
# ============================================================

def dominant_hyper_shap(
    hyper_ids,
    inner_rmse_values
):

    counts = Counter(
        hyper_ids
    )

    max_freq = max(
        counts.values()
    )

    candidates = [
        h
        for h, freq
        in counts.items()
        if freq == max_freq
    ]

    # hanya 1 pemenang
    if len(candidates) == 1:

        dominant = candidates[0]

    # jika seri:
    # pilih mean inner RMSE terkecil
    else:

        mean_rmse = {}

        for h in candidates:

            values = [
                inner_rmse_values[i]

                for i in range(
                    len(hyper_ids)
                )

                if hyper_ids[i] == h
            ]

            mean_rmse[h] = float(
                np.mean(
                    values
                )
            )

        dominant = min(
            candidates,
            key=lambda h:
                mean_rmse[h]
        )

    return dominant, max_freq


# ============================================================
# SELEKSI EV DENGAN SHAP
# SEMUA 99 EV
# ============================================================

def pilih_ev_shap(
    E_train,
    residual_train,
    inner_splits,
    cumulative_threshold=0.90
):

    # --------------------------------------------------------
    # 1. Cari hyperparameter terbaik
    # untuk model selector SHAP
    #
    # target = residual baseline
    # predictor = semua EV
    # --------------------------------------------------------

    selector_best_hp, _ = (
        cari_hyperparameter_terbaik(
            E_train,
            residual_train,
            inner_splits
        )
    )


    # --------------------------------------------------------
    # 2. Fit selector pada seluruh outer-training
    # --------------------------------------------------------

    selector_model = (
        buat_model_xgb(
            selector_best_hp
        )
    )

    selector_model.fit(
        E_train,
        residual_train
    )


    # --------------------------------------------------------
    # 3. Hitung SHAP contribution
    # pred_contribs:
    # 99 EV + 1 bias
    # --------------------------------------------------------

    dtrain = xgb.DMatrix(
        E_train
    )

    shap_values = (
        selector_model
        .get_booster()
        .predict(
            dtrain,
            pred_contribs=True
        )
    )


    # --------------------------------------------------------
    # Kolom terakhir = bias
    # jadi dibuang
    # --------------------------------------------------------

    shap_ev = (
        shap_values[
            :,
            :-1
        ]
    )


    assert shap_ev.shape[1] == (
        E_train.shape[1]
    )


    # --------------------------------------------------------
    # Global importance
    # mean absolute SHAP
    # --------------------------------------------------------

    importance = np.mean(
        np.abs(
            shap_ev
        ),
        axis=0
    )


    total_importance = float(
        np.sum(
            importance
        )
    )


    # --------------------------------------------------------
    # Kalau semuanya nol
    # --------------------------------------------------------

    if total_importance <= 1e-15:

        return (
            np.array(
                [],
                dtype=int
            ),
            selector_best_hp,
            importance
        )


    # --------------------------------------------------------
    # Urutkan importance terbesar
    # --------------------------------------------------------

    order = np.argsort(
        importance
    )[::-1]


    cumulative = (
        np.cumsum(
            importance[
                order
            ]
        )
        /
        total_importance
    )


    # --------------------------------------------------------
    # Ambil EV sampai cumulative >= 90%
    # --------------------------------------------------------

    n_keep = (
        np.searchsorted(
            cumulative,
            cumulative_threshold,
            side="left"
        )
        +
        1
    )


    selected_ev = (
        order[
            :n_keep
        ]
    )


    return (
        np.asarray(
            selected_ev,
            dtype=int
        ),
        selector_best_hp,
        importance
    )


print("✅ HELPER SHAP SIAP")
print("✅ Candidate awal = semua 99 EV")
print(
    "✅ Cumulative threshold =",
    SHAP_CUMULATIVE_THRESHOLD
)

# ============================================================
# ESF-XGBOOST SHAP
# WEAK - GRID10
# ============================================================

for nomor, scenario_id in enumerate(
    SCENARIO_IDS,
    start=1
):

    # ========================================================
    # DATA SCENARIO
    # ========================================================

    df_s = (
        DATA_MODEL[
            DATA_MODEL[
                COL_SCENARIO
            ] == scenario_id
        ]
        .copy()
        .sort_values(
            [
                COL_YCOORD,
                COL_XCOORD
            ]
        )
        .reset_index(drop=True)
    )

    assert len(df_s) == 100


    # ========================================================
    # DATA
    # ========================================================

    X_base = (
        df_s[
            PREDICTOR_COLS
        ]
        .to_numpy(
            dtype=float
        )
    )

    y = (
        df_s[
            COL_TARGET
        ]
        .to_numpy(
            dtype=float
        )
    )

    coords = (
        df_s[
            [
                COL_XCOORD,
                COL_YCOORD
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )


    # ========================================================
    # METADATA
    # ========================================================

    pattern = (
        df_s[
            COL_PATTERN
        ].iloc[0]
    )

    w_raw = (
        df_s[
            COL_W
        ].iloc[0]
    )

    error_type = (
        df_s[
            COL_ERROR
        ].iloc[0]
    )

    rho_value = float(
        df_s[
            COL_RHO
        ].iloc[0]
    )

    w_method = canonical_w(
        w_raw
    )


    # ========================================================
    # BANK ESF
    # SEMUA 99 EV
    # ========================================================

    E = ESF_GRID10[
        w_method
    ]

    assert E.shape == (
        100,
        99
    )


    print(
        f"\n[{nomor:02d}/18] "
        f"{scenario_id}"
    )

    print(
        "   Candidate EV = 99"
    )


    # ========================================================
    # STORAGE SCENARIO
    # ========================================================

    oof_pred = np.full(
        100,
        np.nan
    )

    best_hyper_fold = []

    inner_rmse_fold = []

    n_ev_fold = []

    shap_selector_hyper_fold = []


    # ========================================================
    # 5 OUTER SPATIAL FOLD
    # ========================================================

    for outer_fold in range(
        OUTER_FOLDS
    ):

        # ----------------------------------------------------
        # Outer test
        # ----------------------------------------------------

        test_idx = np.where(
            OUTER_LABELS_GRID10
            ==
            outer_fold
        )[0]


        # ----------------------------------------------------
        # Outer train
        # ----------------------------------------------------

        train_idx = np.where(
            OUTER_LABELS_GRID10
            !=
            outer_fold
        )[0]


        # ====================================================
        # DATA TRAIN TEST
        # ====================================================

        X_train_base = (
            X_base[
                train_idx
            ]
        )

        X_test_base = (
            X_base[
                test_idx
            ]
        )

        y_train = (
            y[
                train_idx
            ]
        )

        y_test = (
            y[
                test_idx
            ]
        )

        coords_train = (
            coords[
                train_idx
            ]
        )

        E_train = (
            E[
                train_idx,
                :
            ]
        )

        E_test = (
            E[
                test_idx,
                :
            ]
        )


        # ====================================================
        # INNER SPATIAL CV
        # ====================================================

        inner_splits = (
            buat_inner_split(
                coords_train,
                n_splits=
                    INNER_FOLDS,
                seed=
                    SEED_MODEL
                    +
                    outer_fold
            )
        )


        # ====================================================
        # 1. BASELINE CROSS-FITTED RESIDUAL
        #
        # Hanya outer-training
        # ====================================================

        residual_train = (
            baseline_crossfit_residual(
                X_train_base,
                y_train,
                coords_train,

                seed=
                    SEED_MODEL
                    +
                    500
                    +
                    outer_fold
            )
        )


        # ====================================================
        # 2. SHAP SELECTION
        #
        # SEMUA 99 EV
        # ====================================================

        (
            selected_ev,
            selector_best_hp,
            shap_importance
        ) = pilih_ev_shap(
            E_train,
            residual_train,
            inner_splits,

            cumulative_threshold=
                SHAP_CUMULATIVE_THRESHOLD
        )


        n_ev = int(
            len(
                selected_ev
            )
        )


        n_ev_fold.append(
            n_ev
        )


        shap_selector_hyper_fold.append(
            selector_best_hp[
                "Hyper_ID"
            ]
        )


        # ====================================================
        # 3. X BASE + EV TERPILIH
        # ====================================================

        if n_ev > 0:

            X_train = np.column_stack([
                X_train_base,

                E_train[
                    :,
                    selected_ev
                ]
            ])


            X_test = np.column_stack([
                X_test_base,

                E_test[
                    :,
                    selected_ev
                ]
            ])

        else:

            X_train = (
                X_train_base.copy()
            )

            X_test = (
                X_test_base.copy()
            )


        # ====================================================
        # 4. H01-H18 UNTUK MODEL FINAL
        #
        # ini berbeda dengan selector SHAP
        # ====================================================

        best_hp, tuning_table = (
            cari_hyperparameter_terbaik(
                X_train,
                y_train,
                inner_splits
            )
        )


        best_hyper_fold.append(
            best_hp[
                "Hyper_ID"
            ]
        )


        inner_rmse_fold.append(
            float(
                best_hp[
                    "Inner_CV_RMSE"
                ]
            )
        )


        # ====================================================
        # 5. FIT FINAL ESF-XGBOOST SHAP
        # ====================================================

        model = buat_model_xgb(
            best_hp
        )

        model.fit(
            X_train,
            y_train
        )


        pred_test = model.predict(
            X_test
        )


        # ====================================================
        # 6. SIMPAN OOF
        # ====================================================

        oof_pred[
            test_idx
        ] = pred_test


        # ====================================================
        # 7. EV LIST
        # index dibuat 1-based agar mudah dibaca
        # ====================================================

        selected_ev_text = ",".join(
            [
                f"EV{i + 1}"
                for i in selected_ev
            ]
        )


        # ====================================================
        # 8. FOLD RESULT
        # ====================================================

        SHAP_FOLD_RESULTS.append({

            "Scenario_ID":
                scenario_id,

            "Grid_Size":
                10,

            "Pattern":
                pattern,

            "W_Method":
                w_method,

            "Error_Structure":
                error_type,

            "Rho":
                rho_value,

            "Model":
                "ESF-XGBoost SHAP",

            "Outer_Fold":
                outer_fold + 1,

            "Candidate_EV":
                99,

            "SHAP_Cumulative_Threshold":
                SHAP_CUMULATIVE_THRESHOLD,

            "N_EV_Selected":
                n_ev,

            "Selected_EV":
                selected_ev_text,

            "SHAP_Selector_Hyper_ID":
                selector_best_hp[
                    "Hyper_ID"
                ],

            "Best_Hyper_ID":
                best_hp[
                    "Hyper_ID"
                ],

            "n_estimators":
                int(
                    best_hp[
                        "n_estimators"
                    ]
                ),

            "learning_rate":
                float(
                    best_hp[
                        "learning_rate"
                    ]
                ),

            "max_depth":
                int(
                    best_hp[
                        "max_depth"
                    ]
                ),

            "Inner_CV_RMSE":
                float(
                    best_hp[
                        "Inner_CV_RMSE"
                    ]
                ),

            "R2_Test":
                float(
                    r2_score(
                        y_test,
                        pred_test
                    )
                ),

            "RMSE_Test":
                float(
                    hitung_rmse(
                        y_test,
                        pred_test
                    )
                ),

            "MAE_Test":
                float(
                    mean_absolute_error(
                        y_test,
                        pred_test
                    )
                )
        })


        print(
            f"   Fold {outer_fold + 1}"
            f" → EV={n_ev:2d}"
            f" | SHAP Selector="
            f"{selector_best_hp['Hyper_ID']}"
            f" | Final="
            f"{best_hp['Hyper_ID']}"
        )


    # ========================================================
    # VALIDASI OOF
    # ========================================================

    assert np.isfinite(
        oof_pred
    ).all()


    # ========================================================
    # OOF METRICS
    # ========================================================

    R2_OOF = float(
        r2_score(
            y,
            oof_pred
        )
    )

    RMSE_OOF = float(
        hitung_rmse(
            y,
            oof_pred
        )
    )

    MAE_OOF = float(
        mean_absolute_error(
            y,
            oof_pred
        )
    )


    # ========================================================
    # HYPERPARAMETER DOMINAN MODEL FINAL
    # ========================================================

    dominant_hyper, max_freq = (
        dominant_hyper_shap(
            best_hyper_fold,
            inner_rmse_fold
        )
    )


    hp_dom = (
        HYPERPARAMETERS[
            HYPERPARAMETERS[
                "Hyper_ID"
            ]
            ==
            dominant_hyper
        ]
        .iloc[0]
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    SHAP_SUMMARY.append({

        "Scenario_ID":
            scenario_id,

        "Grid_Size":
            10,

        "Pattern":
            pattern,

        "W_Method":
            w_method,

        "Error_Structure":
            error_type,

        "Rho":
            rho_value,

        "Model":
            "ESF-XGBoost SHAP",

        "ESF_Selection_Method":
            "SHAP",

        "Candidate_EV":
            99,

        "SHAP_Cumulative_Threshold":
            SHAP_CUMULATIVE_THRESHOLD,

        "SHAP_Selector_Hyper_Fold1":
            shap_selector_hyper_fold[0],

        "SHAP_Selector_Hyper_Fold2":
            shap_selector_hyper_fold[1],

        "SHAP_Selector_Hyper_Fold3":
            shap_selector_hyper_fold[2],

        "SHAP_Selector_Hyper_Fold4":
            shap_selector_hyper_fold[3],

        "SHAP_Selector_Hyper_Fold5":
            shap_selector_hyper_fold[4],

        "Best_Hyper_Fold1":
            best_hyper_fold[0],

        "Best_Hyper_Fold2":
            best_hyper_fold[1],

        "Best_Hyper_Fold3":
            best_hyper_fold[2],

        "Best_Hyper_Fold4":
            best_hyper_fold[3],

        "Best_Hyper_Fold5":
            best_hyper_fold[4],

        "Dominant_Hyper_ID":
            dominant_hyper,

        "Hyper_Frequency":
            f"{max_freq}/5",

        "n_estimators":
            int(
                hp_dom[
                    "n_estimators"
                ]
            ),

        "learning_rate":
            float(
                hp_dom[
                    "learning_rate"
                ]
            ),

        "max_depth":
            int(
                hp_dom[
                    "max_depth"
                ]
            ),

        "N_EV_Fold1":
            n_ev_fold[0],

        "N_EV_Fold2":
            n_ev_fold[1],

        "N_EV_Fold3":
            n_ev_fold[2],

        "N_EV_Fold4":
            n_ev_fold[3],

        "N_EV_Fold5":
            n_ev_fold[4],

        "Mean_N_EV_Selected":
            float(
                np.mean(
                    n_ev_fold
                )
            ),

        "Mean_Inner_CV_RMSE":
            float(
                np.mean(
                    inner_rmse_fold
                )
            ),

        "R2_OOF":
            R2_OOF,

        "RMSE_OOF":
            RMSE_OOF,

        "MAE_OOF":
            MAE_OOF,

        "Status":
            "OK"
    })


    # ========================================================
    # DETAIL PREDICTION
    # ========================================================

    SHAP_PREDICTIONS[
        scenario_id
    ] = pd.DataFrame({

        "Scenario_ID":
            scenario_id,

        "Location_ID":
            (
                df_s[
                    COL_LOCATION
                ].to_numpy()

                if COL_LOCATION
                is not None

                else
                np.arange(
                    1,
                    101
                )
            ),

        "Y_Actual":
            y,

        "Pred_ESF_SHAP":
            oof_pred,

        "Residual_ESF_SHAP":
            y - oof_pred
    })


    print(
        f"   ✅ OOF "
        f"R2={R2_OOF:.4f} | "
        f"RMSE={RMSE_OOF:.4f} | "
        f"MAE={MAE_OOF:.4f} | "
        f"Mean EV="
        f"{np.mean(n_ev_fold):.1f}"
    )


# ============================================================
# FINAL DATAFRAME
# ============================================================

SHAP_SUMMARY_GRID10 = pd.DataFrame(
    SHAP_SUMMARY
)

SHAP_FOLD_GRID10 = pd.DataFrame(
    SHAP_FOLD_RESULTS
)


print("\n" + "=" * 70)
print("🎉 ESF-XGBOOST SHAP GRID10 SELESAI")
print("=" * 70)

print(
    "Summary:",
    SHAP_SUMMARY_GRID10.shape
)

print(
    "Fold:",
    SHAP_FOLD_GRID10.shape
)

print(
    "Predictions:",
    len(
        SHAP_PREDICTIONS
    )
)

✅ HELPER SHAP SIAP
✅ Candidate awal = semua 99 EV
✅ Cumulative threshold = 0.9

[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS
   Candidate EV = 99
   Fold 1 → EV=51 | SHAP Selector=H07 | Final=H10
   Fold 2 → EV=49 | SHAP Selector=H04 | Final=H10
   Fold 3 → EV=57 | SHAP Selector=H09 | Final=H17
   Fold 4 → EV=49 | SHAP Selector=H14 | Final=H10
   Fold 5 → EV=58 | SHAP Selector=H09 | Final=H14
   ✅ OOF R2=0.1729 | RMSE=2.9170 | MAE=2.2074 | Mean EV=52.8

[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS
   Candidate EV = 99
   Fold 1 → EV=56 | SHAP Selector=H05 | Final=H10
   Fold 2 → EV=40 | SHAP Selector=H01 | Final=H02
   Fold 3 → EV=58 | SHAP Selector=H18 | Final=H16
   Fold 4 → EV=53 | SHAP Selector=H03 | Final=H10
   Fold 5 → EV=55 | SHAP Selector=H03 | Final=H13
   ✅ OOF R2=0.1892 | RMSE=2.8546 | MAE=2.1333 | Mean EV=52.4

[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS
   Candidate EV = 99
   Fold 1 → EV=43 | SHAP Selector=H01 | Final=H13
   Fold 2 → EV=59 | SHAP Selector=H14 | Final=

In [51]:
# ============================================================
# VALIDASI SHAP GRID10
# ============================================================

assert len(
    SHAP_SUMMARY_GRID10
) == 18

assert len(
    SHAP_FOLD_GRID10
) == 90

assert len(
    SHAP_PREDICTIONS
) == 18

assert (
    SHAP_SUMMARY_GRID10[
        "Candidate_EV"
    ] == 99
).all()

assert (
    SHAP_SUMMARY_GRID10[
        "R2_OOF"
    ]
    .notna()
    .all()
)

assert (
    SHAP_SUMMARY_GRID10[
        "RMSE_OOF"
    ]
    .notna()
    .all()
)

assert (
    SHAP_SUMMARY_GRID10[
        "MAE_OOF"
    ]
    .notna()
    .all()
)


print("=" * 70)
print("✅ ESF-XGBOOST SHAP GRID10: PASS")
print("=" * 70)

display(
    SHAP_SUMMARY_GRID10[
        [
            "Scenario_ID",
            "W_Method",

            "Candidate_EV",

            "N_EV_Fold1",
            "N_EV_Fold2",
            "N_EV_Fold3",
            "N_EV_Fold4",
            "N_EV_Fold5",

            "Mean_N_EV_Selected",

            "Dominant_Hyper_ID",

            "R2_OOF",
            "RMSE_OOF",
            "MAE_OOF"
        ]
    ]
)

✅ ESF-XGBOOST SHAP GRID10: PASS


,Scenario_ID,W_Method,Candidate_EV,N_EV_Fold1,N_EV_Fold2,N_EV_Fold3,N_EV_Fold4,N_EV_Fold5,Mean_N_EV_Selected,Dominant_Hyper_ID,R2_OOF,RMSE_OOF,MAE_OOF
0,G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS,KNN5,99,51,49,57,49,58,52.8,H10,0.172881,2.916956,2.207403
1,G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS,KNN5,99,56,40,58,53,55,52.4,H10,0.189246,2.854610,2.133255
2,G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,99,43,59,50,44,54,50.0,H13,0.140411,2.969771,2.195916
3,G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,99,54,49,57,58,57,55.0,H14,0.113351,2.981109,2.279955
4,G10_HOTSPOT_IDW_WEAK_HOMOGENEOUS,IDW,99,44,55,60,43,46,49.6,H10,0.113904,2.947959,2.185242
5,G10_HOTSPOT_IDW_WEAK_HETEROGENEOUS,IDW,99,49,56,56,40,55,51.2,H16,0.093924,2.944938,2.268954
6,G10_KORIDOR_KNN5_WEAK_HOMOGENEOUS,KNN5,99,58,53,45,49,52,51.4,H13,0.429056,2.270111,1.814586
7,G10_KORIDOR_KNN5_WEAK_HETEROGENEOUS,KNN5,99,56,45,52,56,56,53.0,H13,0.348207,2.402595,1.964948
8,G10_KORIDOR_DISTANCE_BAND_WEAK_HOMOGENEOUS,DISTANCE,99,55,55,54,51,51,53.2,H11,0.337385,2.446886,1.970514
9,G10_KORIDOR_DISTANCE_BAND_WEAK_HETEROGENEOUS,DISTANCE,99,53,59,48,59,55,54.8,H01,0.378343,2.347425,1.904509


## FILE FINAL GRID 10 X 10 WEAK

In [52]:
# ============================================================
# FINALISASI GRID10 - BAGIAN 1
# MORAN RESIDUAL + GABUNG 5 MODEL
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse


# ============================================================
# CEK SEMUA HASIL MODEL SUDAH ADA
# ============================================================

required_objects = [
    "BASELINE_SUMMARY_GRID10",
    "MORAN_SUMMARY_GRID10",
    "LASSO_SUMMARY_GRID10",
    "STEPWISE_SUMMARY_GRID10",
    "SHAP_SUMMARY_GRID10",

    "BASELINE_FOLD_GRID10",
    "MORAN_FOLD_GRID10",
    "LASSO_FOLD_GRID10",
    "STEPWISE_FOLD_GRID10",
    "SHAP_FOLD_GRID10",

    "BASELINE_PREDICTIONS",
    "MORAN_PREDICTIONS",
    "LASSO_PREDICTIONS",
    "STEPWISE_PREDICTIONS",
    "SHAP_PREDICTIONS",

    "W_GRID10"
]

for obj in required_objects:
    assert obj in globals(), f"❌ {obj} belum tersedia"

print("✅ Semua hasil 5 model tersedia")


# ============================================================
# MORAN'S I RESIDUAL + PERMUTATION P-VALUE
# ============================================================

def moran_residual_test(
    residual,
    W,
    permutations=999,
    seed=2026
):

    residual = np.asarray(
        residual,
        dtype=float
    )

    z = residual - np.mean(residual)

    denominator = float(
        z @ z
    )

    if denominator <= 1e-15:
        return np.nan, np.nan

    n = len(z)

    S0 = float(
        W.sum()
    )

    I_obs = (
        n / S0
    ) * (
        float(
            z @ (W @ z)
        )
        / denominator
    )

    # Expected Moran I
    expected_I = -1.0 / (n - 1)

    observed_distance = abs(
        I_obs - expected_I
    )

    rng = np.random.default_rng(
        seed
    )

    extreme = 0

    for _ in range(permutations):

        zp = rng.permutation(z)

        I_perm = (
            n / S0
        ) * (
            float(
                zp @ (W @ zp)
            )
            /
            float(
                zp @ zp
            )
        )

        if abs(
            I_perm - expected_I
        ) >= observed_distance:
            extreme += 1

    p_value = (
        extreme + 1
    ) / (
        permutations + 1
    )

    return (
        float(I_obs),
        float(p_value)
    )


# ============================================================
# MAP MODEL
# ============================================================

MODEL_INFO = {

    "XGBoost": {
        "summary": BASELINE_SUMMARY_GRID10,
        "predictions": BASELINE_PREDICTIONS,
        "residual_col": "Residual_XGBoost"
    },

    "ESF-XGBoost Moran": {
        "summary": MORAN_SUMMARY_GRID10,
        "predictions": MORAN_PREDICTIONS,
        "residual_col": "Residual_ESF_Moran"
    },

    "ESF-XGBoost LASSO": {
        "summary": LASSO_SUMMARY_GRID10,
        "predictions": LASSO_PREDICTIONS,
        "residual_col": "Residual_ESF_LASSO"
    },

    "ESF-XGBoost Stepwise": {
        "summary": STEPWISE_SUMMARY_GRID10,
        "predictions": STEPWISE_PREDICTIONS,
        "residual_col": "Residual_ESF_Stepwise"
    },

    "ESF-XGBoost SHAP": {
        "summary": SHAP_SUMMARY_GRID10,
        "predictions": SHAP_PREDICTIONS,
        "residual_col": "Residual_ESF_SHAP"
    }
}


# ============================================================
# HITUNG MORAN RESIDUAL
# UNTUK 18 SCENARIO × 5 MODEL
# ============================================================

SUMMARY_WITH_MORAN = []

scenario_ids = (
    DATA_MODEL[COL_SCENARIO]
    .drop_duplicates()
    .tolist()
)


for scenario_no, scenario_id in enumerate(
    scenario_ids,
    start=1
):

    df_s = (
        DATA_MODEL[
            DATA_MODEL[COL_SCENARIO] == scenario_id
        ]
        .copy()
        .sort_values(
            [COL_YCOORD, COL_XCOORD]
        )
        .reset_index(drop=True)
    )

    w_raw = df_s[COL_W].iloc[0]
    w_method = canonical_w(w_raw)

    W = W_GRID10[w_method]

    print(
        f"[{scenario_no:02d}/18] "
        f"{scenario_id}"
    )

    for model_no, (
        model_name,
        info
    ) in enumerate(
        MODEL_INFO.items(),
        start=1
    ):

        pred_df = (
            info["predictions"][
                scenario_id
            ]
        )

        residual = (
            pred_df[
                info["residual_col"]
            ]
            .to_numpy(dtype=float)
        )

        moran_i, moran_p = (
            moran_residual_test(
                residual,
                W,
                permutations=999,
                seed=
                    2026
                    + scenario_no * 100
                    + model_no
            )
        )

        summary_row = (
            info["summary"][
                info["summary"][
                    "Scenario_ID"
                ] == scenario_id
            ]
            .copy()
        )

        assert len(summary_row) == 1

        summary_row[
            "Moran_I_Residual_OOF"
        ] = moran_i

        summary_row[
            "Moran_pvalue_Residual_OOF"
        ] = moran_p

        summary_row[
            "Residual_Spatial_Status"
        ] = (
            "Tidak Signifikan"
            if moran_p > 0.05
            else
            "Signifikan"
        )

        SUMMARY_WITH_MORAN.append(
            summary_row
        )

        print(
            f"   {model_name:25s} "
            f"I={moran_i: .4f} | "
            f"p={moran_p:.3f}"
        )


# ============================================================
# SUMMARY FINAL 90 BARIS
# ============================================================

SUMMARY_FINAL_GRID10 = (
    pd.concat(
        SUMMARY_WITH_MORAN,
        ignore_index=True
    )
)


MODEL_ORDER = [
    "XGBoost",
    "ESF-XGBoost Moran",
    "ESF-XGBoost LASSO",
    "ESF-XGBoost Stepwise",
    "ESF-XGBoost SHAP"
]

SUMMARY_FINAL_GRID10[
    "Model"
] = pd.Categorical(
    SUMMARY_FINAL_GRID10["Model"],
    categories=MODEL_ORDER,
    ordered=True
)

SUMMARY_FINAL_GRID10 = (
    SUMMARY_FINAL_GRID10
    .sort_values(
        [
            "Scenario_ID",
            "Model"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# GABUNG FOLD EVALUATION
# 18 × 5 × 5 = 450 BARIS
# ============================================================

FOLD_FINAL_GRID10 = pd.concat(
    [
        BASELINE_FOLD_GRID10,
        MORAN_FOLD_GRID10,
        LASSO_FOLD_GRID10,
        STEPWISE_FOLD_GRID10,
        SHAP_FOLD_GRID10
    ],
    ignore_index=True,
    sort=False
)


# ============================================================
# GABUNG DETAIL PREDICTION
# 1 DATAFRAME PER SCENARIO
# ============================================================

DETAIL_FINAL_GRID10 = {}


for scenario_id in scenario_ids:

    # baseline sebagai dasar
    detail = (
        BASELINE_PREDICTIONS[
            scenario_id
        ]
        .copy()
    )

    # pastikan koordinat ikut ada
    df_s = (
        DATA_MODEL[
            DATA_MODEL[COL_SCENARIO]
            == scenario_id
        ]
        .copy()
        .sort_values(
            [COL_YCOORD, COL_XCOORD]
        )
        .reset_index(drop=True)
    )

    # Tambahkan koordinat jika belum ada
    if "X_norm" not in detail.columns:
        detail["X_norm"] = (
            df_s[COL_XCOORD]
            .to_numpy()
        )

    if "Y_norm" not in detail.columns:
        detail["Y_norm"] = (
            df_s[COL_YCOORD]
            .to_numpy()
        )

    # Outer fold
    if "Outer_Fold" not in detail.columns:
        detail["Outer_Fold"] = (
            OUTER_LABELS_GRID10 + 1
        )


    # --------------------------------------------------------
    # Moran
    # --------------------------------------------------------

    moran_df = MORAN_PREDICTIONS[
        scenario_id
    ]

    detail[
        "Pred_ESF_Moran"
    ] = moran_df[
        "Pred_ESF_Moran"
    ].to_numpy()

    detail[
        "Residual_ESF_Moran"
    ] = moran_df[
        "Residual_ESF_Moran"
    ].to_numpy()


    # --------------------------------------------------------
    # LASSO
    # --------------------------------------------------------

    lasso_df = LASSO_PREDICTIONS[
        scenario_id
    ]

    detail[
        "Pred_ESF_LASSO"
    ] = lasso_df[
        "Pred_ESF_LASSO"
    ].to_numpy()

    detail[
        "Residual_ESF_LASSO"
    ] = lasso_df[
        "Residual_ESF_LASSO"
    ].to_numpy()


    # --------------------------------------------------------
    # Stepwise
    # --------------------------------------------------------

    step_df = STEPWISE_PREDICTIONS[
        scenario_id
    ]

    detail[
        "Pred_ESF_Stepwise"
    ] = step_df[
        "Pred_ESF_Stepwise"
    ].to_numpy()

    detail[
        "Residual_ESF_Stepwise"
    ] = step_df[
        "Residual_ESF_Stepwise"
    ].to_numpy()


    # --------------------------------------------------------
    # SHAP
    # --------------------------------------------------------

    shap_df = SHAP_PREDICTIONS[
        scenario_id
    ]

    detail[
        "Pred_ESF_SHAP"
    ] = shap_df[
        "Pred_ESF_SHAP"
    ].to_numpy()

    detail[
        "Residual_ESF_SHAP"
    ] = shap_df[
        "Residual_ESF_SHAP"
    ].to_numpy()


    # --------------------------------------------------------
    # Urutan kolom
    # --------------------------------------------------------

    desired_cols = [
        "Scenario_ID",
        "Location_ID",
        "X_norm",
        "Y_norm",
        "Y_Actual",
        "Outer_Fold",

        "Pred_XGBoost",
        "Residual_XGBoost",

        "Pred_ESF_Moran",
        "Residual_ESF_Moran",

        "Pred_ESF_LASSO",
        "Residual_ESF_LASSO",

        "Pred_ESF_Stepwise",
        "Residual_ESF_Stepwise",

        "Pred_ESF_SHAP",
        "Residual_ESF_SHAP"
    ]

    detail = detail[
        desired_cols
    ]

    DETAIL_FINAL_GRID10[
        scenario_id
    ] = detail


print("\n" + "=" * 70)
print("✅ MORAN RESIDUAL + GABUNG 5 MODEL SELESAI")
print("=" * 70)

print(
    "Summary rows :",
    len(SUMMARY_FINAL_GRID10)
)

print(
    "Fold rows    :",
    len(FOLD_FINAL_GRID10)
)

print(
    "Detail sheet :",
    len(DETAIL_FINAL_GRID10)
)

✅ Semua hasil 5 model tersedia
[01/18] G10_HOTSPOT_KNN5_WEAK_HOMOGENEOUS
   XGBoost                   I= 0.3264 | p=0.001
   ESF-XGBoost Moran         I= 0.3744 | p=0.001
   ESF-XGBoost LASSO         I= 0.3797 | p=0.001
   ESF-XGBoost Stepwise      I= 0.4094 | p=0.001
   ESF-XGBoost SHAP          I= 0.3734 | p=0.001
[02/18] G10_HOTSPOT_KNN5_WEAK_HETEROGENEOUS
   XGBoost                   I= 0.3471 | p=0.001
   ESF-XGBoost Moran         I= 0.3715 | p=0.001
   ESF-XGBoost LASSO         I= 0.3751 | p=0.001
   ESF-XGBoost Stepwise      I= 0.4376 | p=0.001
   ESF-XGBoost SHAP          I= 0.3646 | p=0.001
[03/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HOMOGENEOUS
   XGBoost                   I= 0.3240 | p=0.001
   ESF-XGBoost Moran         I= 0.3861 | p=0.001
   ESF-XGBoost LASSO         I= 0.3397 | p=0.001
   ESF-XGBoost Stepwise      I= 0.3768 | p=0.001
   ESF-XGBoost SHAP          I= 0.3854 | p=0.001
[04/18] G10_HOTSPOT_DISTANCE_BAND_WEAK_HETEROGENEOUS
   XGBoost                   I= 0.3403 | p=0

In [53]:
# ============================================================
# FINALISASI GRID10 - BAGIAN 2
# BUAT 2 EXCEL + VALIDASI
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# OUTPUT FOLDER
# ============================================================

OUTPUT_DIR = Path(
    "/content/FILE3_WEAK"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FILE_DETAIL = (
    OUTPUT_DIR
    /
    "HASIL_WEAK_GRID10.xlsx"
)

FILE_SUMMARY = (
    OUTPUT_DIR
    /
    "SUMMARY_WEAK_GRID10.xlsx"
)


# ============================================================
# HELPER NAMA SHEET
# ============================================================

def sheet_name_grid10(
    scenario_id
):

    df_s = (
        DATA_MODEL[
            DATA_MODEL[
                COL_SCENARIO
            ] == scenario_id
        ]
        .iloc[0]
    )

    pattern = str(
        df_s[COL_PATTERN]
    ).upper()

    w_raw = str(
        df_s[COL_W]
    ).upper()

    error = str(
        df_s[COL_ERROR]
    ).upper()


    # Pattern
    if "HOT" in pattern:
        p = "HOT"

    elif (
        "COR" in pattern
        or
        "KOR" in pattern
    ):
        p = "KOR"

    else:
        p = "GEL"


    # W
    if "KNN" in w_raw:
        w = "KNN"

    elif "IDW" in w_raw:
        w = "IDW"

    else:
        w = "DIST"


    # Error
    if (
        "HETER" in error
        or
        "HET" in error
    ):
        e = "HET"

    else:
        e = "HOMO"


    return f"{p}_{w}_{e}"


# ============================================================
# 1. HASIL_WEAK_GRID10.xlsx
# 18 SHEET
# ============================================================

with pd.ExcelWriter(
    FILE_DETAIL,
    engine="openpyxl"
) as writer:

    for scenario_id in scenario_ids:

        sheet = sheet_name_grid10(
            scenario_id
        )

        DETAIL_FINAL_GRID10[
            scenario_id
        ].to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

        ws = writer.book[
            sheet
        ]

        ws.freeze_panes = "A2"

        ws.auto_filter.ref = (
            ws.dimensions
        )


# ============================================================
# 2. SUMMARY_WEAK_GRID10.xlsx
# ============================================================

with pd.ExcelWriter(
    FILE_SUMMARY,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # SUMMARY 90 BARIS
    # --------------------------------------------------------

    SUMMARY_FINAL_GRID10.to_excel(
        writer,
        sheet_name="SUMMARY_MODEL",
        index=False
    )


    # --------------------------------------------------------
    # DETAIL PER FOLD
    # --------------------------------------------------------

    FOLD_FINAL_GRID10.to_excel(
        writer,
        sheet_name="FOLD_EVALUATION",
        index=False
    )


    # --------------------------------------------------------
    # 18 KOMBINASI HYPERPARAMETER
    # --------------------------------------------------------

    HYPERPARAMETERS.to_excel(
        writer,
        sheet_name="HYPERPARAMETERS",
        index=False
    )


    # --------------------------------------------------------
    # SETTINGS
    # --------------------------------------------------------

    SETTINGS_GRID10 = pd.DataFrame({

        "Setting": [
            "Spatial Strength",
            "Rho",
            "Grid",
            "N Location",
            "Outer Spatial CV",
            "Inner Spatial CV",
            "Hyperparameter Combination",
            "Moran Residual Permutations",
            "SHAP Cumulative Threshold",
            "Stepwise Selection"
        ],

        "Value": [
            "Weak",
            0.20,
            "10x10",
            100,
            5,
            3,
            18,
            999,
            0.90,
            "Forward Spatial-CV RMSE"
        ]
    })


    SETTINGS_GRID10.to_excel(
        writer,
        sheet_name="SETTINGS",
        index=False
    )


    # --------------------------------------------------------
    # Freeze panes
    # --------------------------------------------------------

    for ws in writer.book.worksheets:

        ws.freeze_panes = "A2"

        ws.auto_filter.ref = (
            ws.dimensions
        )


# ============================================================
# VALIDASI FILE
# ============================================================

assert FILE_DETAIL.exists()
assert FILE_SUMMARY.exists()

assert len(
    SUMMARY_FINAL_GRID10
) == 90

assert len(
    FOLD_FINAL_GRID10
) == 450

assert len(
    DETAIL_FINAL_GRID10
) == 18


# Moran residual harus lengkap
assert (
    SUMMARY_FINAL_GRID10[
        "Moran_I_Residual_OOF"
    ]
    .notna()
    .all()
)

assert (
    SUMMARY_FINAL_GRID10[
        "Moran_pvalue_Residual_OOF"
    ]
    .notna()
    .all()
)


# R2 RMSE MAE harus lengkap
for col in [
    "R2_OOF",
    "RMSE_OOF",
    "MAE_OOF"
]:

    assert (
        SUMMARY_FINAL_GRID10[
            col
        ]
        .notna()
        .all()
    )


# Setiap scenario 5 model
cek_model = (
    SUMMARY_FINAL_GRID10
    .groupby(
        "Scenario_ID",
        observed=True
    )
    .size()
)

assert (
    cek_model == 5
).all()


# Setiap detail 100 lokasi
for scenario_id, df_detail in (
    DETAIL_FINAL_GRID10.items()
):

    assert len(df_detail) == 100


print("=" * 70)
print("🎉 GRID10 FINAL VALIDATION: PASS")
print("=" * 70)

print(
    "\nFILE 1:",
    FILE_DETAIL.name
)

print(
    "Ukuran:",
    round(
        FILE_DETAIL.stat().st_size
        /
        1024**2,
        2
    ),
    "MB"
)

print(
    "\nFILE 2:",
    FILE_SUMMARY.name
)

print(
    "Ukuran:",
    round(
        FILE_SUMMARY.stat().st_size
        /
        1024**2,
        2
    ),
    "MB"
)

print(
    "\nSummary :",
    SUMMARY_FINAL_GRID10.shape
)

print(
    "Fold    :",
    FOLD_FINAL_GRID10.shape
)

print(
    "Sheets  :",
    len(
        DETAIL_FINAL_GRID10
    )
)

print("\n✅ 2 FILE GRID10 SIAP DIUPLOAD")

🎉 GRID10 FINAL VALIDATION: PASS

FILE 1: HASIL_WEAK_GRID10.xlsx
Ukuran: 0.29 MB

FILE 2: SUMMARY_WEAK_GRID10.xlsx
Ukuran: 0.09 MB

Summary : (90, 40)
Fold    : (450, 22)
Sheets  : 18

✅ 2 FILE GRID10 SIAP DIUPLOAD


In [59]:
# ============================================================
# LOGIN GITHUB TANPA PAT/TOKEN MANUAL
# LALU PUSH COMMIT GRID10 YANG SUDAH ADA
# ============================================================

import shutil
import subprocess


# ============================================================
# 1. INSTALL GITHUB CLI JIKA BELUM ADA
# ============================================================

if shutil.which("gh") is None:

    print("Menginstall GitHub CLI...")

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )

    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "gh"],
        check=True
    )

print("✅ GitHub CLI tersedia")


# ============================================================
# 2. CEK APA SUDAH LOGIN
# ============================================================

status = subprocess.run(
    ["gh", "auth", "status"],
    capture_output=True,
    text=True
)


if status.returncode != 0:

    print("\n🔐 Login GitHub diperlukan")
    print("Ikuti petunjuk yang muncul.")
    print("Biasanya akan muncul kode dan link GitHub.\n")

    # Login via browser/device flow
    subprocess.run(
        [
            "gh",
            "auth",
            "login",
            "--hostname",
            "github.com",
            "--git-protocol",
            "https",
            "--web"
        ],
        check=True
    )

else:

    print("✅ GitHub sudah login")


# ============================================================
# 3. HUBUNGKAN AUTH GH KE GIT
# ============================================================

subprocess.run(
    [
        "gh",
        "auth",
        "setup-git"
    ],
    check=True
)

print("✅ Git authentication siap")


# ============================================================
# 4. PUSH COMMIT GRID10 YANG SUDAH ADA
# ============================================================

subprocess.run(
    [
        "git",
        "push",
        "origin",
        "main"
    ],
    cwd=REPO_DIR,
    check=True
)


# ============================================================
# 5. CEK STATUS
# ============================================================

status_git = subprocess.run(
    [
        "git",
        "status",
        "-sb"
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=True
)

print("\n" + "=" * 70)
print("🎉 PUSH GRID10 BERHASIL")
print("=" * 70)

print(status_git.stdout)

print("File:")
print("results/file3_weak/HASIL_WEAK_GRID10.xlsx")
print("results/file3_weak/SUMMARY_WEAK_GRID10.xlsx")

✅ GitHub CLI tersedia

🔐 Login GitHub diperlukan
Ikuti petunjuk yang muncul.
Biasanya akan muncul kode dan link GitHub.



KeyboardInterrupt: 